# Setup

In [1]:
from datetime import date, datetime
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from pathlib import Path
from time import sleep
import subprocess
import requests
import pandas
import spacy
import glob
import time
import json
import csv
import re
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # go to ~/crossdem/ by jumping up twice
DATA_DIR = os.path.join(BASE_DIR, "datasets")
PMS_DIR = os.path.join(DATA_DIR, "prime_ministers")
VALIDATION_DIR = os.path.join(DATA_DIR, "prime_ministers_validation")
IMGS_DIR = os.path.join(os.getcwd(), "imgs")

os.makedirs(IMGS_DIR, exist_ok=True)

# De Gasperi was not scraped, a dataset with all speeches was taken from https://github.com/StefanoMenini/De-Gasperi-s-Corpus/
#degasperi_df = pandas.read_csv(f"{DATA_DIR}/degasperi/degasperi_speeches.csv")

POL_INFO = {
    "degasperi":   {"leaning": "C",  "color": "#2f4f4f"},  # dark slate grey — PM from 1945
    "fanfani":     {"leaning": "CL", "color": "#e6194B"},  # red — 1954
    #"scelba":      {"leaning": "CR", "color": "#8b4513"},  # saddle brown — 1954 (no data)
    #"segni":       {"leaning": "CR", "color": "#1e90ff"},  # dodger blue — 1955 (no data)
    "leone":       {"leaning": "C",  "color": "#ff4500"},  # orange-red — 1963
    #"moro":        {"leaning": "CL", "color": "#556b2f"},  # dark olive green — 1963 (no data)
    "rumor":       {"leaning": "C",  "color": "#7f0067"},  # deep magenta/plum — 1968
    "colombo":     {"leaning": "C",  "color": "#008080"},  # dark teal — 1970
    "andreotti":   {"leaning": "CR", "color": "#c00000"},  # bright crimson — 1972
    "cossiga":     {"leaning": "CR", "color": "#e6beff"},  # light violet — 1979
    "forlani":     {"leaning": "CR", "color": "#ffe119"},  # yellow — 1980
    "spadolini":   {"leaning": "C",  "color": "#a9a9a9"},  # grey — 1981
    "craxi":       {"leaning": "CL", "color": "#ffd8b1"},  # apricot — 1983
    "goria":       {"leaning": "C",  "color": "#000075"},  # navy — 1987
    "demita":     {"leaning": "CL", "color": "#808000"},  # olive — 1988
    "amato":       {"leaning": "L",  "color": "#aaffc3"},  # mint — 1992
    "ciampi":      {"leaning": "C",  "color": "#9A6324"},  # brown — 1993
    "berlusconi":  {"leaning": "R",  "color": "#800000"},  # dark red / maroon — 1994
    "dini":        {"leaning": "CL", "color": "#fabed4"},  # pink — 1995
    "prodi":       {"leaning": "CL", "color": "#469990"},  # teal — 1996
    "dalema":     {"leaning": "L",  "color": "#dcbeff"},  # lavender — 1998
    "monti":       {"leaning": "CR", "color": "#bfef45"},  # lime — 2011
    "letta":       {"leaning": "L",  "color": "#f032e6"},  # magenta — 2013
    "renzi":       {"leaning": "CL", "color": "#42d4f4"},  # cyan — 2014
    "gentiloni":   {"leaning": "CL", "color": "#911eb4"},  # purple — 2016
    "conte":       {"leaning": "CL", "color": "#f58231"},  # orange — 2018
    "draghi":      {"leaning": "C",  "color": "#3cb44b"},  # green — 2021
    "meloni":      {"leaning": "R",  "color": "#4363d8"},  # blue — 2022
}

DISPLAY_NAME_OVERRIDES = {
    "dalema": "D'Alema",
    "demita": "De Mita",
    "degasperi": "De Gasperi"
}

def display_name(pol):
    return DISPLAY_NAME_OVERRIDES.get(pol, pol.capitalize())

CHECK_INTEGRITY = False

# backward compatibility
#POLITICIANS = [
#    pol for pol in POL_INFO
#    if pol != "degasperi" and pol not in ("moro", "segni", "scelba")
#]
#POL_COLORS = {pol: info["color"] for pol, info in POL_INFO.items()}
#POL_LEANING = {pol: info["leaning"] for pol, info in POL_INFO.items()}

#politicians_dfs = {
#    "degasperi": degasperi_df
#}

#for pol in politicians:
#    speeches = glob.glob(f"{DATA_DIR}/{pol}/csv_out/*.csv")
#    pol_df = pandas.concat([pandas.read_csv(f, nrows=1) for f in speeches], ignore_index=True)
#    politicians_dfs[pol] = pol_df

politicians_dfs = {}
# csv faster for smaller datasets (eg 1 row)
def read_single_row(path):
    with open(path, newline="", encoding="utf-8") as f:
        return next(csv.DictReader(f))

for pol in POL_INFO:
    speeches = glob.glob(f"{PMS_DIR}/{pol}/csv_out/*.csv")
    rows = [read_single_row(f) for f in speeches]
    if pol == "degasperi":
        df1 = pandas.read_csv(f"{PMS_DIR}/degasperi/degasperi_speeches.csv")
        df2 = pandas.read_csv(f"{VALIDATION_DIR}/degasperi/degasperi_validation.csv")
        politicians_dfs[pol] = pandas.concat([df1, df2])
    elif pol == "meloni":
        df1 = pandas.DataFrame(rows)
        df2 = pandas.read_csv(f"{VALIDATION_DIR}/meloni/meloni_validation.csv")
        politicians_dfs[pol] = pandas.concat([df1, df2])
    else:
        politicians_dfs[pol] = pandas.DataFrame(rows)
#print(politicians_dfs["conte"])
politicians_dfs["meloni"].head()

,politician,historical_date,location,title,url,audio_file,text,tags,description,hate_speech,negativity,aggressiveness,target
0,meloni,2000-10-16,ROMA,"""Convegno di Destra Protagonista a Roma"" org. ...",https://www.radioradicale.it/scheda/124586/con...,crossdem/datasets/meloni/audio_out/124586_melo...,"Ci sia una vittoria, una vittoria netta, una v...",NaN,NaN,NaN,NaN,NaN,NaN
1,meloni,2000-11-22,ROMA,Incontro tra gli studenti del Liceo Torquato T...,https://www.radioradicale.it/scheda/133778/inc...,crossdem/datasets/meloni/audio_out/133778_melo...,"di richiesto e di chiarimento che voglio fare,...",NaN,NaN,NaN,NaN,NaN,NaN
2,meloni,2001-12-19,ROMA,Stati Generali dell'Istruzione (presso il Pala...,https://www.radioradicale.it/scheda/134365/sta...,crossdem/datasets/meloni/audio_out/134365_melo...,caratteristiche delle verifiche nazionali già ...,NaN,NaN,NaN,NaN,NaN,NaN
3,meloni,2000-12-16,VERONA,"""Quid est Veritas? Verità, cultura, testo scol...",https://www.radioradicale.it/scheda/135795/qui...,crossdem/datasets/meloni/audio_out/135795_melo...,è quello di inventare un'autorità che tuteli e...,NaN,NaN,NaN,NaN,NaN,NaN
4,meloni,2002-02-16,ROMA,Convention di Destra protagonista componente ...,https://www.radioradicale.it/scheda/136073/con...,crossdem/datasets/meloni/audio_out/136073_melo...,coordinatrice del Comitato per il mondo giovan...,NaN,NaN,NaN,NaN,NaN,NaN



# Sentiment Analysis

## Type Of Architecture Choice

Encoder-Decoder models are often trained on Twitter corpus -> dont perform well on political speeches.
Also BERT cannot handle more than 512 tokens in one go, I have avg 2k token per speech.

My data is *unlabelled*, therefore a model like BERT which relies a lot on fine-tuning is not the right choice. On the other hand, decoder only models don't need this phase at all.

Oss: LLMs are bad ad continuous scales, better to as for score between 1 and 5.



## Find High Negativity Examples

In [ ]:
"""
find_hate_negativity_candidates.py

Lexicon-based triage over the full ~6k-speech corpus (PMS_DIR) to surface
candidate speeches worth hand-labeling, instead of hand-labeling at random.

Scores every already-scraped speech (PMS_DIR/{politician}/csv_out/*.csv) for:
  - negativity: density of negative-sentiment lemmas
  - hate_speech: density of sentences where an ethnic/religious/gender GROUP
                 term co-occurs with a HOSTILITY term (same sentence)

Copies the top N speeches per dimension (raw CSV files, unmodified, exactly
as they exist in csv_out/) into VALIDATION_DIR/{politician}/, so you only
need to hand-label a small, high-yield pool instead of the full corpus.

IMPORTANT — read before trusting the output:
- This is a coarse triage filter, not a hate-speech detector. The lexicons
  below are small seed lists I wrote quickly. They WILL miss things, and
  they WILL surface false positives — e.g. a speech *condemning* xenophobia
  mentions both a group term and a hostility term in the same sentence and
  will score just as high as a speech being hostile itself. You still need
  to read and hand-label whatever gets copied here; this step only improves
  which 15+15 speeches you spend that time on.
- For something more validated than a hand-typed seed list, look at HurtLex
  (Bassignana et al.) — an academically-vetted Italian hostility lexicon —
  as a drop-in replacement for HOSTILITY_LEMMAS.
- Speeches already present in VALIDATION_DIR/{politician}/ are excluded from
  candidacy so you don't re-flag something already labeled.

Requires BASE_DIR / PMS_DIR / VALIDATION_DIR / POL_INFO already in scope
(from your crossdem setup cell).

Install deps:
    pip install spacy pandas tqdm --break-system-packages
    python -m spacy download it_core_news_sm
"""

import os
import glob
import shutil
from pathlib import Path

import pandas as pd
import spacy
from tqdm import tqdm

TEXT_COL = "text"   # <-- CHECK: should match TEXT_COL in compare_llm_classifiers.py
TOP_N = 15

CACHE_DIR = Path(os.path.join(BASE_DIR, "source/pred_cache"))
os.makedirs(CACHE_DIR, exist_ok=True)

# --------------------------------------------------------------------------- #
# Seed lexicons — small starting point, extend/replace as needed
# --------------------------------------------------------------------------- #

NEGATIVITY_LEMMAS = {
    "crisi", "fallimento", "pericolo", "minaccia", "nemico", "corruzione",
    "disastro", "vergogna", "tradimento", "odio", "violenza", "morte", "guerra",
    "catastrofe", "degrado", "declino", "sconfitta", "ingiustizia", "povertà",
    "disoccupazione", "insicurezza", "paura", "terrore", "allarme", "emergenza",
    "scandalo", "illegalità", "criminalità", "invasione", "distruzione",
    "rovina", "collasso", "disperazione", "indignazione", "rabbia", "inganno",
    "menzogna", "bugia", "ipocrisia", "colpa", "condanna", "attacco", "conflitto",
}

# group-reference terms — deliberately neutral/descriptive, not slurs
GROUP_LEMMAS = {
    "immigrato", "straniero", "migrante", "rom", "sinto", "africano", "arabo",
    "cinese", "albanese", "rumeno", "nero",
    "musulmano", "islamico", "ebreo",
    "donna", "omosessuale", "gay", "lesbica", "transgender",
}

HOSTILITY_LEMMAS = {
    "invasione", "minaccia", "pericolo", "criminale", "delinquente", "clandestino",
    "cacciare", "espellere", "rimandare", "inferiore", "sporco", "sporcare",
    "nemico", "invadere", "aggredire", "delinquere",
}

# --------------------------------------------------------------------------- #

print("Loading spaCy model...")
nlp = spacy.load("it_core_news_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")


def score_text(doc) -> dict:
    n_tokens = len(doc)
    if n_tokens == 0:
        return {"negativity_score": 0.0, "hate_score": 0.0, "n_tokens": 0}

    neg_hits = sum(1 for tok in doc if tok.lemma_.lower() in NEGATIVITY_LEMMAS)
    negativity_score = 1000 * neg_hits / n_tokens

    hate_sentence_hits = 0
    n_sents = 0
    for sent in doc.sents:
        n_sents += 1
        lemmas = {tok.lemma_.lower() for tok in sent}
        if lemmas & GROUP_LEMMAS and lemmas & HOSTILITY_LEMMAS:
            hate_sentence_hits += 1
    hate_score = 1000 * hate_sentence_hits / max(n_sents, 1)

    return {"negativity_score": negativity_score, "hate_score": hate_score, "n_tokens": n_tokens}


def collect_candidate_files() -> list:
    """Every scraped speech file not already sitting in VALIDATION_DIR/{pol}/."""
    candidates = []
    for pol in POL_INFO:
        pol_dir = os.path.join(PMS_DIR, pol, "csv_out")
        files = glob.glob(os.path.join(pol_dir, "*.csv"))
        if not files:
            continue  # e.g. degasperi, which has no per-speech csv_out files

        dest_dir = os.path.join(VALIDATION_DIR, pol)
        os.makedirs(dest_dir, exist_ok=True)
        already_validated = {Path(f).name for f in glob.glob(os.path.join(dest_dir, "*.csv"))}

        for f in files:
            if Path(f).name in already_validated:
                continue
            candidates.append({"politician": pol, "filepath": f})
    return candidates


def copy_to_validation(df: pd.DataFrame, dimension: str):
    for _, row in df.iterrows():
        src = Path(row["filepath"])
        dest_dir = Path(VALIDATION_DIR) / row["politician"]
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / src.name
        if dest.exists():
            print(f"  [{dimension}] already present, skipping: {dest}")
            continue
        shutil.copy2(src, dest)
        print(f"  [{dimension}] copied {src.name} -> {dest_dir}")


def main():
    candidates = collect_candidate_files()
    print(f"Found {len(candidates)} candidate speeches not already in VALIDATION_DIR.")

    texts = []
    valid_candidates = []
    for c in candidates:
        try:
            row = pd.read_csv(c["filepath"]).iloc[0]
        except Exception as e:
            print(f"  skipping {c['filepath']}: {e}")
            continue
        if TEXT_COL not in row or not isinstance(row[TEXT_COL], str):
            continue
        texts.append(row[TEXT_COL])
        valid_candidates.append(c)

    print(f"Scoring {len(valid_candidates)} speeches (this can take a few minutes for ~5k speeches)...")
    for c, doc in tqdm(zip(valid_candidates, nlp.pipe(texts, batch_size=32)), total=len(valid_candidates)):
        c.update(score_text(doc))

    scores_df = pd.DataFrame(valid_candidates)
    scores_path = CACHE_DIR / "candidate_scores.csv"
    scores_df.to_csv(scores_path, index=False)
    print(f"Full ranking (all {len(scores_df)} candidates) saved to {scores_path}")

    top_negativity = scores_df.sort_values("negativity_score", ascending=False).head(TOP_N)
    top_hate = scores_df.sort_values("hate_score", ascending=False).head(TOP_N)

    print(f"\nTop {TOP_N} negativity candidates:")
    print(top_negativity[["politician", "filepath", "negativity_score", "n_tokens"]].to_string(index=False))

    print(f"\nTop {TOP_N} hate_speech candidates:")
    print(top_hate[["politician", "filepath", "hate_score", "n_tokens"]].to_string(index=False))

    copy_to_validation(top_negativity, "negativity")
    copy_to_validation(top_hate, "hate_speech")


if __name__ == "__main__":
    pass
    #main()

Loading spaCy model...
Found 5465 candidate speeches not already in VALIDATION_DIR.
Scoring 5464 speeches (this can take a few minutes for ~5k speeches)...


100%|██████████| 5464/5464 [07:24<00:00, 12.30it/s]

Full ranking (all 5464 candidates) saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/candidate_scores.csv

Top 15 negativity candidates:
politician                                                                                                    filepath  negativity_score  n_tokens
      dini            /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/dini/csv_out/314276_dini_s2t.csv         37.500000       160
     letta          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/letta/csv_out/118109_letta_s2t.csv         26.490066       151
 andreotti  /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/andreotti/csv_out/222504_andreotti_s2t.csv         22.598870       177
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/264004_dalema_s2t.csv         22.222222       225
     prodi          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/prodi/csv_out/119887_prodi_s

Output: 

```bash
Loading spaCy model...
Found 5465 candidate speeches not already in VALIDATION_DIR.
Scoring 5464 speeches (this can take a few minutes for ~5k speeches)...
100%|██████████| 5464/5464 [07:24<00:00, 12.30it/s]Full ranking (all 5464 candidates) saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/candidate_scores.csv

Top 15 negativity candidates:
politician                                                                                                    filepath  negativity_score  n_tokens
      dini            /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/dini/csv_out/314276_dini_s2t.csv         37.500000       160
     letta          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/letta/csv_out/118109_letta_s2t.csv         26.490066       151
 andreotti  /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/andreotti/csv_out/222504_andreotti_s2t.csv         22.598870       177
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/264004_dalema_s2t.csv         22.222222       225
     prodi          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/prodi/csv_out/119887_prodi_s2t.csv         21.782178       505
     prodi          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/prodi/csv_out/219002_prodi_s2t.csv         19.607843        51
    meloni /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/meloni/csv_out/sMzNn_meloni_speech2text.csv         18.599562       914
     letta          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/letta/csv_out/161521_letta_s2t.csv         18.255578       493
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/134219_dalema_s2t.csv         17.994859       389
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/211407_dalema_s2t.csv         17.490494      2630
 andreotti  /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/andreotti/csv_out/158786_andreotti_s2t.csv         17.139090      1517
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/203971_dalema_s2t.csv         16.806723       238
     renzi          /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/renzi/csv_out/433686_renzi_s2t.csv         16.233766       308
 andreotti  /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/andreotti/csv_out/223670_andreotti_s2t.csv         15.904573       503
   d'alema       /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/149151_dalema_s2t.csv         15.846258      2966

Top 15 hate_speech candidates:
politician                                                                                                     filepath  hate_score  n_tokens
     amato           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/amato/csv_out/136189_amato_s2t.csv      1000.0      2797
     amato           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/amato/csv_out/204629_amato_s2t.csv      1000.0      2062
     amato           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/amato/csv_out/137542_amato_s2t.csv      1000.0      5747
   d'alema        /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/d'alema/csv_out/655295_dalema_s2t.csv      1000.0      1999
berlusconi /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/berlusconi/csv_out/156460_berlusconi_s2t.csv      1000.0      5765
    meloni         /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/meloni/csv_out/217051_meloni_s2t.csv      1000.0      2708
berlusconi /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/berlusconi/csv_out/228170_berlusconi_s2t.csv      1000.0      1314
berlusconi /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/berlusconi/csv_out/204646_berlusconi_s2t.csv      1000.0      7892
     letta           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/letta/csv_out/319099_letta_s2t.csv      1000.0      2639
    meloni         /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/meloni/csv_out/331612_meloni_s2t.csv      1000.0      3072
    meloni         /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/meloni/csv_out/316761_meloni_s2t.csv      1000.0      1258
 andreotti   /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/andreotti/csv_out/245886_andreotti_s2t.csv      1000.0      3891
     monti           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/monti/csv_out/144957_monti_s2t.csv      1000.0      2308
     amato           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/amato/csv_out/257502_amato_s2t.csv      1000.0      5687
     amato           /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/amato/csv_out/456686_amato_s2t.csv      1000.0      1953
  [negativity] copied 314276_dini_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/dini
  [negativity] copied 118109_letta_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/letta
  [negativity] copied 222504_andreotti_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/andreotti
  [negativity] copied 264004_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [negativity] copied 119887_prodi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/prodi
  [negativity] copied 219002_prodi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/prodi
  [negativity] copied sMzNn_meloni_speech2text.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/meloni
  [negativity] copied 161521_letta_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/letta
  [negativity] copied 134219_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [negativity] copied 211407_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [negativity] copied 158786_andreotti_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/andreotti
  [negativity] copied 203971_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [negativity] copied 433686_renzi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/renzi
  [negativity] copied 223670_andreotti_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/andreotti
  [negativity] copied 149151_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [hate_speech] copied 136189_amato_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/amato
  [hate_speech] copied 204629_amato_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/amato
  [hate_speech] copied 137542_amato_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/amato
  [hate_speech] copied 655295_dalema_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/d'alema
  [hate_speech] copied 156460_berlusconi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/berlusconi
  [hate_speech] copied 217051_meloni_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/meloni
  [hate_speech] copied 228170_berlusconi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/berlusconi
  [hate_speech] copied 204646_berlusconi_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/berlusconi
  [hate_speech] copied 319099_letta_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/letta
  [hate_speech] copied 331612_meloni_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/meloni
  [hate_speech] copied 316761_meloni_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/meloni
  [hate_speech] copied 245886_andreotti_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/andreotti
  [hate_speech] copied 144957_monti_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/monti
  [hate_speech] copied 257502_amato_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/amato
  [hate_speech] copied 456686_amato_s2t.csv -> /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers_validation/amato
```

## Evaluate LLMs Classification Models

### Thinking  OFF
Overall best models are *gemma4:e4b* for hate speech, negativity, and aggressiveness; and *qwen3.5:4b* for target. 

In [2]:
"""
compare_llm_classifiers.py

Compares mistralai/Mistral-7B-Instruct-v0.3 vs Qwen/Qwen2.5-7B-Instruct (both run
locally via Ollama, quantized) on the crossdem 4-label classification task:

    hate_speech     : low | mid | high
    negativity      : low | mid | high
    aggressiveness  : low | mid | high
    target          : none | pol_adv | minor_etn | minor_gnd | minor_rel

Validation set = your 100 hand-labeled speeches (degasperi_validation.csv +
meloni_validation.csv).

Assumes BASE_DIR / VALIDATION_DIR are already defined in your environment
(e.g. from your crossdem setup cell/module) before this script/cell runs.

Design notes:
- Uses Ollama's JSON-schema `format` param for grammar-constrained decoding, same
  spirit as your existing Qwen/llama.cpp pipeline.
- Predictions are cached to CSV incrementally (one API call = one row appended),
  so an interrupted run resumes instead of re-classifying from scratch.
- temperature=0 for reproducibility.
- VERBOSE=True prints, for every call: the row id, the text fed to the model,
  the raw model output, and the parsed prediction.

REQUIRES YOU TO CHECK / EDIT:
1. TEXT_COL, ID_COL below.
2. OLLAMA model tags — verify with `ollama list`.
3. The RUBRIC text in SYSTEM_PROMPT should match your manual annotation rubric.

Install deps:
    pip install ollama scikit-learn pandas
"""

import os
import json
import time
from pathlib import Path

import pandas as pd
from ollama import Client
from sklearn.metrics import (
    cohen_kappa_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# --------------------------------------------------------------------------- #
# CONFIG — check this section before running
# --------------------------------------------------------------------------- #
POL_VAL = [
    "degasperi",     
    "amato",        
    "prodi",          
    "letta",          
    "meloni",
    "synthetic"
]

VALIDATION_FILES = []

for pol in POL_VAL:
    speeches = glob.glob(f"{VALIDATION_DIR}/{pol}/*.csv")
    for speech in speeches:
        VALIDATION_FILES.append(speech)


TEXT_COL = "text"          # <-- CHECK: column holding the speech transcript
ID_COL = None               # <-- CHECK: unique id column, or leave None to use row index

ORDINAL_COLS = ["hate_speech", "negativity", "aggressiveness"]
NOMINAL_COLS = ["target"]
ALL_LABEL_COLS = ORDINAL_COLS + NOMINAL_COLS  # also the fixed output order for the pred lists

LEVEL_VALUES = ["low", "mid", "high"]
TARGET_VALUES = ["none", "pol_adv", "minor_etn", "minor_gnd", "minor_rel"]

# 4 bit quantization
OLLAMA_MODELS = {
    "qwen2.5-7b": "qwen2.5:7b-instruct-q4_K_M",       # <-- CHECK exact local tag
    "mistral-7b-v0.3": "mistral:7b-instruct-v0.3-q4_K_M",  # <-- CHECK exact local tag
    "qwen3.5-4b": "qwen3.5:4b-q8_0",
    "qwen3.5-9b": "qwen3.5:9b-q4_K_M",
    "gemma4-e4b": "gemma4:e4b",
    "gemma4-12b": "gemma4:12b",
    "qwen3.5-4b-stock": "qwen3.5:4b",
}

CACHE_DIR = Path(os.path.join(BASE_DIR, "source/pred_cache"))
os.makedirs(CACHE_DIR, exist_ok=True)

RETRIES = 0
RETRY_SLEEP_S = 2

# --- verbosity ---
VERBOSE = True
PRINT_TEXT_CHARS = 400  # set to None to print the full speech text every call

client = Client()  # assumes `ollama serve` is running locally

# --------------------------------------------------------------------------- #
# PROMPT — align this with your manual annotation rubric
# --------------------------------------------------------------------------- #


SYSTEM_PROMPT = """Sei un annotatore esperto di scienze politiche e linguistica computazionale.
Il tuo compito è classificare un discorso pronunciato da un Presidente del Consiglio italiano secondo quattro dimensioni.

---
### DIMENSIONI E CATEGORIE

1) hate_speech: presenza di incitamento all'odio verso individui o gruppi protetti.
   - low: assente o non significativa.
   - mid: presente in forma velata, allusiva, tramite "dog whistles" o pregiudizi impliciti.
   - high: presente in forma esplicita, deumanizzante o aperta ostilità sistematica.

2) negativity: tono complessivo negativo del discorso (critica, pessimismo, minacce percepite, allarmismo).
   - low: tono prevalentemente neutro, costruttivo o positivo.
   - mid: presenza moderata di toni critici o allarmi su specifici problemi nazionali.
   - high: tono fortemente cupo, allarmista, focalizzato su crisi imminenti o declino.

3) aggressiveness: aggressività verbale, toni polemici o ostili verso interlocutori/avversari.
   - low: tono pacato, formale, istituzionale.
   - mid: presenza moderata di polemica, velate stoccate o sarcasmo istituzionale.
   - high: tono marcatamente polemico, attacchi diretti e personali, linguaggio di scontro.

4) target: il bersaglio principale di eventuale ostilità/critica nel discorso (assegna il target anche se i punteggi sopra sono "low", identificando verso chi è orientato il discorso).
   - none: nessun bersaglio specifico identificabile.
   - pol_adv: avversari politici, opposizioni, partiti, burocrazia europea o altre istituzioni.
   - minor_etn: minoranze etniche, persone straniere, migranti di specifiche etnie.
   - minor_gnd: minoranze di genere, donne, comunità LGBTQ+.
   - minor_rel: minoranze religiose (es. musulmani, ebrei).

---
### ESEMPI GUIDA PER LA CALIBRAZIONE

#### Esempi per `hate_speech`:
- **low**:
  - "Dobbiamo investire nella scuola pubblica e garantire il salario minimo per sostenere le famiglie in difficoltà."
  - "La transizione ecologica richiede una visione industriale chiara che tuteli l'occupazione e le piccole imprese."
  - "Rafforzeremo il presidio del territorio e la cooperazione tra le forze dell'ordine e le comunità locali."
- **mid**:
  - "Non possiamo permettere che i valori della nostra civiltà vengano diluiti da chi arriva da fuori senza alcuna intenzione di assimilarsi."
  - "Certe culture tradizionaliste restano intrinsecamente incompatibili con la tutela dei diritti fondamentali e dello Stato di diritto."
  - "I quartieri storici stanno perdendo la propria identità a causa di una presenza straniera ormai dominante che rifiuta le nostre regole."
- **high**:
  - "L'immigrazione è un'invasione pianificata che porterà degrado e criminalità nelle nostre città. Bisogna eliminarla sistematicamente strada per strada, casa per casa."
  - "Questi gruppi parassitari infestano la nostra società e vanno estirpati prima che distruggano definitivamente la nostra razza e la nostra storia."
  - "Alcune minoranze sono portatrici biologiche di violenza e inciviltà: vanno cacciate con ogni mezzo dal nostro suolo patrio."

#### Esempi per `negativity`:
- **low**:
  - "I dati sull'occupazione sono incoraggianti e la crescita economica dimostra la resilienza del nostro tessuto produttivo."
  - "La riforma della giustizia procede spedita, restituendo efficienza e tempi certi a cittadini e imprese."
  - "Il nostro posizionamento internazionale si rafforza grazie a nuove partnership strategiche sull'energia."
- **mid**:
  - "I sistemi sanitario e scolastico presentano problematiche infrastrutturali apparentemente insanabili."
  - "L'instabilità geopolitica globale proietta ombre preoccupanti sull'approvvigionamento delle nostre materie prime."
  - "La burocrazia soffocante e i tempi della giustizia civile continuano a frenare gli investimenti esteri nel Paese."
- **high**:
  - "Il paese è sull'orlo del baratro finanziario; ereditiamo un disastro sistemico che rischia di spazzare via i risparmi degli italiani."
  - "Siamo di fronte a un declino demografico e sociale irreversibile che sta portando la Nazione alla completa estinzione."
  - "Il collasso della sicurezza urbana ha trasformato le nostre metropoli in zone di guerra allo sbaraglio."

#### Esempi per `aggressiveness`:
- **low**:
  - "Accogliamo le osservazioni delle opposizioni nel merito, tuttavia il Governo continuerà con la linea dettata dagli elettori."
  - "Il confronto parlamentare è la sede naturale per affinare i testi di legge nell'interesse generale."
  - "Valuteremo con attenzione tutti gli emendamenti proposti dalle forze di minoranza durante il percorso in commissione."
- **mid**:
  - "Riconosco il diritto dell'opposizione di protestare, anche se la loro memoria storica appare alquanto corta e di comodo."
  - "Ci danno lezioni di rigore di bilancio gli stessi banchieri e tecnocrati che hanno affossato i conti pubblici nel decennio scorso."
  - "Spiace constatare come parte dei media preferisca la polemica strumentale all'analisi obiettiva dei fatti."
- **high**:
  - "Dall'opposizione arrivano solo menzogne sfrontate e sciacallaggio politico da parte di chi ha svenduto la nazione per anni."
  - "Siete dei traditori del popolo italiano, dei cospiratori servili che prendono ordini da potenze e burocrazie straniere."
  - "La vostra ipocrisia fa schifo: avete le mani sporche di sangue per le politiche criminali che avete approvato!"

#### Esempi per `target`:
- **none**:
  - "Oggi approviamo la riforma del codice della strada per ridurre gli incidenti e tutelare i giovani."
  - "Il piano di digitalizzazione della pubblica amministrazione consentirà di ridurre radicalmente i tempi di attesa."
  - "I finanziamenti per la prevenzione del dissesto idrogeologico copriranno tutte le regioni a rischio."
- **pol_adv**:
  - "La precedente maggioranza ha lasciato buchi di bilancio incalcolabili per fare propaganda elettorale."
  - "L'Unione Europea pretende di imporre direttive ideologiche che penalizzano le nostre filiere produttive nazionali."
  - "La magistratura politicamente orientata continua a travalicare i propri confini costituzionali per ostacolare l'esecutivo."
- **minor_etn**:
  - "È necessario bloccare le partenze e combattere le reti di clandestinità che destabilizzano la sicurezza nazionale."
  - "Certi flussi migratori incontrollati provenienti dall'Africa subsahariana importano modelli criminali inaccettabili."
  - "Non tollereremo zone di franchigia gestite da bande etniche straniere nelle periferie dei nostri capoluoghi."
- **minor_gnd**:
  - "Ci opporremo a chi vuole scardinare la famiglia naturale imponendo teorie ideologiche nelle scuole."
  - "La propaganda sull'identità di genere mira a cancellare la figura della madre e il ruolo biologico della donna."
  - "I diritti delle donne vengono calpestati dall'ossessione per il politicamente corretto e la fluidità di genere."
- **minor_rel**:
  - "Alcune comunità religiose pretendono di applicare le proprie leggi teocratiche sul nostro territorio nazionale."
  - "Il proliferare di centri di preghiera abusivi legati all'Islam radicale rappresenta una minaccia diretta ai nostri valori laici."
  - "Non faremo concessioni a chi usa il proprio culto per giustificare la sottomissione femminile e l'odio verso l'Occidente."

---
### FORMATO DI OUTPUT

Rispondi ESCLUSIVAMENTE con un oggetto JSON con queste quattro chiavi, senza testo
aggiuntivo, markdown o spiegazioni."""

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "hate_speech": {"type": "string", "enum": LEVEL_VALUES},
        "negativity": {"type": "string", "enum": LEVEL_VALUES},
        "aggressiveness": {"type": "string", "enum": LEVEL_VALUES},
        "target": {"type": "string", "enum": TARGET_VALUES},
    },
    "required": ["hate_speech", "negativity", "aggressiveness", "target"],
}


# --------------------------------------------------------------------------- #
# Data loading
# --------------------------------------------------------------------------- #

def load_validation_data() -> pd.DataFrame:
    dfs = [pd.read_csv(f) for f in VALIDATION_FILES]
    df = pd.concat(dfs, ignore_index=True)

    print(f"Loaded {len(df)} validation speeches. Columns: {df.columns.tolist()}")

    missing = [c for c in ALL_LABEL_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Gold label columns missing from validation CSVs: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )
    if TEXT_COL not in df.columns:
        raise ValueError(
            f"TEXT_COL='{TEXT_COL}' not found. Available columns: {df.columns.tolist()}"
        )

    if ID_COL is None:
        df = df.reset_index(drop=True)
        df["_row_id"] = df.index.astype(str)
    else:
        df["_row_id"] = df[ID_COL].astype(str)

    return df


# --------------------------------------------------------------------------- #
# Classification
# --------------------------------------------------------------------------- #

def classify_one(text: str, model_tag: str, row_id: str) -> dict | None:
    for attempt in range(1, RETRIES + 1):
        if VERBOSE:
            preview = text if PRINT_TEXT_CHARS is None else text[:PRINT_TEXT_CHARS]
            truncated_note = "" if PRINT_TEXT_CHARS is None or len(text) <= PRINT_TEXT_CHARS else " [truncated]"
            print(f"\n{'=' * 80}")
            print(f"[{model_tag}] row_id={row_id}  attempt {attempt}/{RETRIES}")
            print(f"--- PROMPT FED TO MODEL ({len(text)} chars{truncated_note}) ---")
            print(preview)

        try:
            resp = client.chat(
                model=model_tag,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                format=JSON_SCHEMA,
                options={"temperature": 0},
                think=False,
            )
            raw = resp["message"]["content"]

            if VERBOSE:
                print(f"--- RAW MODEL OUTPUT ---\n{raw}")

            parsed = json.loads(raw)
            # validate values are in-range; raises if not
            assert parsed["hate_speech"] in LEVEL_VALUES
            assert parsed["negativity"] in LEVEL_VALUES
            assert parsed["aggressiveness"] in LEVEL_VALUES
            assert parsed["target"] in TARGET_VALUES

            if VERBOSE:
                print(f"--- PARSED PREDICTION --- {parsed}")

            return parsed
        except Exception as e:
            print(f"  [{model_tag}] row_id={row_id} attempt {attempt}/{RETRIES} failed: {e}")
            time.sleep(RETRY_SLEEP_S)
    return None


def run_model(model_name: str, model_tag: str, df: pd.DataFrame) -> pd.DataFrame:
    cache_path = Path(f"{CACHE_DIR}/{model_name}_predictions.csv")

    if cache_path.exists():
        cached = pd.read_csv(cache_path, dtype={"_row_id": str})
        done_ids = set(cached["_row_id"])
    else:
        cached = pd.DataFrame(columns=["_row_id"] + ALL_LABEL_COLS)
        done_ids = set()

    rows_to_do = df[~df["_row_id"].isin(done_ids)]
    print(f"\n[{model_name}] {len(done_ids)} cached, {len(rows_to_do)} to classify.")

    new_rows = []
    for i, row in rows_to_do.iterrows():

        # RUN MODEL
        result = classify_one(row[TEXT_COL], model_tag, row["_row_id"])
        print(f"--- GOLD LABELS ---       {{'hate_speech': '{row['hate_speech']}', 'negativity': '{row['negativity']}', 'aggressiveness': '{row['aggressiveness']}', 'target': '{row['target']}'}}")

        if result is None:
            print(f"  [{model_name}] giving up on row {row['_row_id']} after {RETRIES} retries.")
            continue
        result["_row_id"] = row["_row_id"]
        new_rows.append(result)

        # append incrementally so a crash doesn't lose progress
        pd.DataFrame([result]).to_csv(
            cache_path, mode="a", header=not cache_path.exists(), index=False
        )

    if new_rows:
        cached = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)

    return cached


# --------------------------------------------------------------------------- #
# Predictions as a plain dict of lists
# --------------------------------------------------------------------------- #

def build_prediction_lists(pred_df: pd.DataFrame) -> list:
    """
    Turns a predictions dataframe into a plain list of
    [hate_speech, negativity, aggressiveness, target] lists,
    ordered by _row_id (numeric order if row ids are numeric, else lexicographic).

    Example output for one model:
        [
            ["low", "low", "low", "none"],
            ["low", "mid", "low", "none"],
            ...
        ]
    """
    df_sorted = pred_df.copy()
    try:
        df_sorted["_sort_key"] = df_sorted["_row_id"].astype(int)
    except ValueError:
        df_sorted["_sort_key"] = df_sorted["_row_id"]
    df_sorted = df_sorted.sort_values("_sort_key")
    return df_sorted[ALL_LABEL_COLS].values.tolist()


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #

def evaluate(gold: pd.DataFrame, pred: pd.DataFrame, model_name: str) -> dict:
    merged = gold.merge(pred, on="_row_id", suffixes=("_gold", "_pred"))
    print(f"\n=== {model_name}: evaluated on {len(merged)}/{len(gold)} speeches ===")

    results = {}
    for col in ALL_LABEL_COLS:
        y_true = merged[f"{col}_gold"]
        y_pred = merged[f"{col}_pred"]

        weights = "quadratic" if col in ORDINAL_COLS else None
        kappa = cohen_kappa_score(y_true, y_pred, weights=weights)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        acc = (y_true == y_pred).mean()

        results[col] = {
            "cohen_kappa": kappa,
            "accuracy": acc,
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
        }

        print(f"\n--- {col} ---")
        print(f"  Cohen's kappa ({'quadratic-weighted' if weights else 'unweighted'}): {kappa:.3f}")
        print(f"  Accuracy: {acc:.3f}  |  Macro P/R/F1: {precision:.3f} / {recall:.3f} / {f1:.3f}")
        print(classification_report(y_true, y_pred, zero_division=0))
        labels = LEVEL_VALUES if col in ORDINAL_COLS else TARGET_VALUES
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print(f"  Confusion matrix (rows=gold, cols=pred), labels={labels}")
        print(pd.DataFrame(cm, index=labels, columns=labels))

    return results


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    df = load_validation_data()
    gold = df[["_row_id"] + ALL_LABEL_COLS].copy()

    all_results = {}
    predictions_dict = {}

    for model_name, model_tag in OLLAMA_MODELS.items():
        pred = run_model(model_name, model_tag, df)
        predictions_dict[model_name] = build_prediction_lists(pred)
        all_results[model_name] = evaluate(gold, pred, model_name)

    # --- plain dict of lists, e.g. predictions_dict["qwen2.5-7b"][0] == ["low","low","low","none"] ---
    #print("\n\n===== PREDICTIONS (dict of lists, order = [hate_speech, negativity, aggressiveness, target]) =====")
    #for model_name, lists in predictions_dict.items():
    #    print(f"\n{model_name}: {len(lists)} speeches")
    #    for row in lists:
    #        print(row)

    predictions_json_path = CACHE_DIR / "predictions_dict.json"
    with open(predictions_json_path, "w", encoding="utf-8") as f:
        json.dump(predictions_dict, f, ensure_ascii=False, indent=2)
    print(f"\nPredictions dict saved to {predictions_json_path}")

    # summary table across both models
    summary_rows = []
    for model_name, res in all_results.items():
        for col, metrics in res.items():
            summary_rows.append({"model": model_name, "label": col, **metrics})

    summary_df = pd.DataFrame(summary_rows)
    summary_path = CACHE_DIR / "comparison_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n\n===== SUMMARY (Mistral vs Qwen) =====")
    print(summary_df.pivot(index="label", columns="model", values="cohen_kappa").round(3))
    print(f"\nFull summary saved to {summary_path}")

    return predictions_dict, all_results


if __name__ == "__main__":
    main()

Loaded 207 validation speeches. Columns: ['politician', 'historical_date', 'location', 'keywords', 'text', 'hate_speech', 'negativity', 'aggressiveness', 'target', 'title', 'url', 'audio_file', 'tags', 'description']

[qwen2.5-7b] 207 cached, 0 to classify.

=== qwen2.5-7b: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.173
  Accuracy: 0.696  |  Macro P/R/F1: 0.634 / 0.622 / 0.584
              precision    recall  f1-score   support

        high       0.54      0.77      0.63        48
         low       0.80      0.92      0.86       106
         mid       0.56      0.17      0.26        53

    accuracy                           0.70       207
   macro avg       0.63      0.62      0.58       207
weighted avg       0.68      0.70      0.65       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    98    0     8
mid    20    9    24
high    4    7    37

--- negativity ---
  Cohen's 

#### Prompt Refining

##### First Prompt

```python
SYSTEM_PROMPT = """Sei un annotatore esperto di scienze politiche e linguistica computazionale.
Il tuo compito è classificare un discorso pronunciato da un Presidente del Consiglio
italiano secondo quattro dimensioni.

1) hate_speech: presenza di incitamento all'odio verso individui o gruppi.
   - low: assente o non significativa
   - mid: presente in forma velata, allusiva o isolata
   - high: presente in forma esplicita, ripetuta o centrale nel discorso

2) negativity: tono complessivo negativo del discorso (critica, pessimismo, minacce percepite).
   - low: tono prevalentemente neutro o positivo
   - mid: presenza moderata di toni negativi o critici
   - high: tono prevalentemente o fortemente negativo

3) aggressiveness: aggressività verbale, toni polemici o ostili verso interlocutori/avversari.
   - low: tono pacato, istituzionale
   - mid: presenza moderata di polemica o ostilità
   - high: tono marcatamente polemico, ostile o conflittuale

4) target: il bersaglio principale di eventuale ostilità/critica nel discorso (indipendente
   dai punteggi sopra: assegna il target anche se hate_speech/negativity/aggressiveness sono "low").
   - none: nessun bersaglio specifico
   - pol_adv: avversari politici, partiti, altre istituzioni politiche
   - minor_etn: minoranze etniche (es. persone nere, immigrati di specifiche etnie)
   - minor_gnd: minoranze di genere (es. donne, comunità LGBT)
   - minor_rel: minoranze religiose (es. musulmani, ebrei)

Rispondi ESCLUSIVAMENTE con un oggetto JSON con queste quattro chiavi, senza testo
aggiuntivo, markdown o spiegazioni."""
```

```bash
Loaded 207 validation speeches. Columns: ['politician', 'historical_date', 'location', 'keywords', 'text', 'hate_speech', 'negativity', 'aggressiveness', 'target', 'title', 'url', 'audio_file', 'tags', 'description']

[qwen2.5-7b] 207 cached, 0 to classify.

=== qwen2.5-7b: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.038
  Accuracy: 0.556  |  Macro P/R/F1: 0.648 / 0.412 / 0.374
              precision    recall  f1-score   support

        high       0.37      0.21      0.27        48
         low       0.57      0.95      0.72       106
         mid       1.00      0.08      0.14        53

    accuracy                           0.56       207
   macro avg       0.65      0.41      0.37       207
weighted avg       0.64      0.56      0.46       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low   101    0     5
mid    37    4    12
high   38    0    10

--- negativity ---
  Cohen's kappa (quadratic-weighted): 0.118
  Accuracy: 0.522  |  Macro P/R/F1: 0.534 / 0.583 / 0.512
              precision    recall  f1-score   support

        high       0.48      0.83      0.61        66
         low       0.61      0.74      0.67        50
         mid       0.52      0.18      0.26        91

    accuracy                           0.52       207
   macro avg       0.53      0.58      0.51       207
weighted avg       0.53      0.52      0.47       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    37   12     1
mid    16   16    59
high    8    3    55

--- aggressiveness ---
  Cohen's kappa (quadratic-weighted): 0.424
  Accuracy: 0.705  |  Macro P/R/F1: 0.751 / 0.702 / 0.693
              precision    recall  f1-score   support

        high       0.90      0.48      0.63        56
         low       0.65      0.98      0.78        65
         mid       0.71      0.64      0.67        86

    accuracy                           0.71       207
   macro avg       0.75      0.70      0.69       207
weighted avg       0.74      0.71      0.69       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    64    0     1
mid    29   55     2
high    6   23    27

--- target ---
  Cohen's kappa (unweighted): 0.592
  Accuracy: 0.700  |  Macro P/R/F1: 0.755 / 0.705 / 0.721
              precision    recall  f1-score   support

   minor_etn       0.88      0.88      0.88        25
   minor_gnd       0.67      0.40      0.50        25
   minor_rel       0.95      0.80      0.87        25
        none       0.66      0.81      0.72        52
     pol_adv       0.62      0.64      0.63        80

    accuracy                           0.70       207
   macro avg       0.76      0.71      0.72       207
weighted avg       0.71      0.70      0.70       207

  Confusion matrix (rows=gold, cols=pred), labels=['none', 'pol_adv', 'minor_etn', 'minor_gnd', 'minor_rel']
           none  pol_adv  minor_etn  minor_gnd  minor_rel
none         42        8          0          2          0
pol_adv      22       51          3          3          1
minor_etn     0        3         22          0          0
minor_gnd     0       15          0         10          0
minor_rel     0        5          0          0         20

[mistral-7b-v0.3] 207 cached, 0 to classify.

=== mistral-7b-v0.3: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.041
  Accuracy: 0.710  |  Macro P/R/F1: 0.462 / 0.629 / 0.525
              precision    recall  f1-score   support

        high       0.51      0.92      0.66        48
         low       0.87      0.97      0.92       106
         mid       0.00      0.00      0.00        53

    accuracy                           0.71       207
   macro avg       0.46      0.63      0.53       207
weighted avg       0.57      0.71      0.62       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low   103    3     0
mid    11    0    42
high    4    0    44

--- negativity ---
  Cohen's kappa (quadratic-weighted): 0.042
  Accuracy: 0.348  |  Macro P/R/F1: 0.286 / 0.361 / 0.296
              precision    recall  f1-score   support

        high       0.44      0.83      0.57        66
         low       0.22      0.14      0.17        50
         mid       0.20      0.11      0.14        91

    accuracy                           0.35       207
   macro avg       0.29      0.36      0.30       207
weighted avg       0.28      0.35      0.29       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low     7   36     7
mid    17   10    64
high    8    3    55

--- aggressiveness ---
  Cohen's kappa (quadratic-weighted): 0.032
  Accuracy: 0.464  |  Macro P/R/F1: 0.453 / 0.523 / 0.444
              precision    recall  f1-score   support

        high       0.41      0.89      0.56        56
         low       0.68      0.58      0.63        65
         mid       0.27      0.09      0.14        86

    accuracy                           0.46       207
   macro avg       0.45      0.52      0.44       207
weighted avg       0.44      0.46      0.41       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    38   21     6
mid    13    8    65
high    5    1    50

--- target ---
  Cohen's kappa (unweighted): 0.746
  Accuracy: 0.807  |  Macro P/R/F1: 0.887 / 0.889 / 0.879
              precision    recall  f1-score   support

   minor_etn       1.00      1.00      1.00        25
   minor_gnd       1.00      1.00      1.00        25
   minor_rel       1.00      1.00      1.00        25
        none       0.58      0.85      0.69        52
     pol_adv       0.86      0.60      0.71        80

    accuracy                           0.81       207
   macro avg       0.89      0.89      0.88       207
weighted avg       0.84      0.81      0.81       207

  Confusion matrix (rows=gold, cols=pred), labels=['none', 'pol_adv', 'minor_etn', 'minor_gnd', 'minor_rel']
           none  pol_adv  minor_etn  minor_gnd  minor_rel
none         44        8          0          0          0
pol_adv      32       48          0          0          0
minor_etn     0        0         25          0          0
minor_gnd     0        0          0         25          0
minor_rel     0        0          0          0         25

Predictions dict saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/predictions_dict.json


===== SUMMARY (Mistral vs Qwen) =====
model           mistral-7b-v0.3  qwen2.5-7b
label                                      
aggressiveness            0.032       0.424
hate_speech               0.041       0.038
negativity                0.042       0.118
target                    0.746       0.592

Full summary saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/comparison_summary.csv
```

##### Second Prompt

```python
SYSTEM_PROMPT = """Sei un annotatore esperto di scienze politiche e linguistica computazionale.
Il tuo compito è classificare un discorso pronunciato da un Presidente del Consiglio italiano secondo quattro dimensioni.

---
### DIMENSIONI E CATEGORIE

1) hate_speech: presenza di incitamento all'odio verso individui o gruppi protetti.
   - low: assente o non significativa.
   - mid: presente in forma velata, allusiva, tramite "dog whistles" o pregiudizi impliciti.
   - high: presente in forma esplicita, deumanizzante o aperta ostilità sistematica.

2) negativity: tono complessivo negativo del discorso (critica, pessimismo, minacce percepite, allarmismo).
   - low: tono prevalentemente neutro, costruttivo o positivo.
   - mid: presenza moderata di toni critici o allarmi su specifici problemi nazionali.
   - high: tono fortemente cupo, allarmista, focalizzato su crisi imminenti o declino.

3) aggressiveness: aggressività verbale, toni polemici o ostili verso interlocutori/avversari.
   - low: tono pacato, formale, istituzionale.
   - mid: presenza moderata di polemica, velate stoccate o sarcasmo istituzionale.
   - high: tono marcatamente polemico, attacchi diretti e personali, linguaggio di scontro.

4) target: il bersaglio principale di eventuale ostilità/critica nel discorso (assegna il target anche se i punteggi sopra sono "low", identificando verso chi è orientato il discorso).
   - none: nessun bersaglio specifico identificabile.
   - pol_adv: avversari politici, opposizioni, partiti, burocrazia europea o altre istituzioni.
   - minor_etn: minoranze etniche, persone straniere, migranti di specifiche etnie.
   - minor_gnd: minoranze di genere, donne, comunità LGBTQ+.
   - minor_rel: minoranze religiose (es. musulmani, ebrei).

---
### ESEMPI GUIDA PER LA CALIBRAZIONE

#### Esempi per `hate_speech`:
- **low**:
  - "Dobbiamo investire nella scuola pubblica e garantire il salario minimo per sostenere le famiglie in difficoltà."
  - "La transizione ecologica richiede una visione industriale chiara che tuteli l'occupazione e le piccole imprese."
  - "Rafforzeremo il presidio del territorio e la cooperazione tra le forze dell'ordine e le comunità locali."
- **mid**:
  - "Non possiamo permettere che i valori della nostra civiltà vengano diluiti da chi arriva da fuori senza alcuna intenzione di assimilarsi."
  - "Certe culture tradizionaliste restano intrinsecamente incompatibili con la tutela dei diritti fondamentali e dello Stato di diritto."
  - "I quartieri storici stanno perdendo la propria identità a causa di una presenza straniera ormai dominante che rifiuta le nostre regole."
- **high**:
  - "L'immigrazione è un'invasione pianificata che porterà degrado e criminalità nelle nostre città. Bisogna eliminarla sistematicamente strada per strada, casa per casa."
  - "Questi gruppi parassitari infestano la nostra società e vanno estirpati prima che distruggano definitivamente la nostra razza e la nostra storia."
  - "Alcune minoranze sono portatrici biologiche di violenza e inciviltà: vanno cacciate con ogni mezzo dal nostro suolo patrio."

#### Esempi per `negativity`:
- **low**:
  - "I dati sull'occupazione sono incoraggianti e la crescita economica dimostra la resilienza del nostro tessuto produttivo."
  - "La riforma della giustizia procede spedita, restituendo efficienza e tempi certi a cittadini e imprese."
  - "Il nostro posizionamento internazionale si rafforza grazie a nuove partnership strategiche sull'energia."
- **mid**:
  - "I sistemi sanitario e scolastico presentano problematiche infrastrutturali apparentemente insanabili."
  - "L'instabilità geopolitica globale proietta ombre preoccupanti sull'approvvigionamento delle nostre materie prime."
  - "La burocrazia soffocante e i tempi della giustizia civile continuano a frenare gli investimenti esteri nel Paese."
- **high**:
  - "Il paese è sull'orlo del baratro finanziario; ereditiamo un disastro sistemico che rischia di spazzare via i risparmi degli italiani."
  - "Siamo di fronte a un declino demografico e sociale irreversibile che sta portando la Nazione alla completa estinzione."
  - "Il collasso della sicurezza urbana ha trasformato le nostre metropoli in zone di guerra allo sbaraglio."

#### Esempi per `aggressiveness`:
- **low**:
  - "Accogliamo le osservazioni delle opposizioni nel merito, tuttavia il Governo continuerà con la linea dettata dagli elettori."
  - "Il confronto parlamentare è la sede naturale per affinare i testi di legge nell'interesse generale."
  - "Valuteremo con attenzione tutti gli emendamenti proposti dalle forze di minoranza durante il percorso in commissione."
- **mid**:
  - "Riconosco il diritto dell'opposizione di protestare, anche se la loro memoria storica appare alquanto corta e di comodo."
  - "Ci danno lezioni di rigore di bilancio gli stessi banchieri e tecnocrati che hanno affossato i conti pubblici nel decennio scorso."
  - "Spiace constatare come parte dei media preferisca la polemica strumentale all'analisi obiettiva dei fatti."
- **high**:
  - "Dall'opposizione arrivano solo menzogne sfrontate e sciacallaggio politico da parte di chi ha svenduto la nazione per anni."
  - "Siete dei traditori del popolo italiano, dei cospiratori servili che prendono ordini da potenze e burocrazie straniere."
  - "La vostra ipocrisia fa schifo: avete le mani sporche di sangue per le politiche criminali che avete approvato!"

#### Esempi per `target`:
- **none**:
  - "Oggi approviamo la riforma del codice della strada per ridurre gli incidenti e tutelare i giovani."
  - "Il piano di digitalizzazione della pubblica amministrazione consentirà di ridurre radicalmente i tempi di attesa."
  - "I finanziamenti per la prevenzione del dissesto idrogeologico copriranno tutte le regioni a rischio."
- **pol_adv**:
  - "La precedente maggioranza ha lasciato buchi di bilancio incalcolabili per fare propaganda elettorale."
  - "L'Unione Europea pretende di imporre direttive ideologiche che penalizzano le nostre filiere produttive nazionali."
  - "La magistratura politicamente orientata continua a travalicare i propri confini costituzionali per ostacolare l'esecutivo."
- **minor_etn**:
  - "È necessario bloccare le partenze e combattere le reti di clandestinità che destabilizzano la sicurezza nazionale."
  - "Certi flussi migratori incontrollati provenienti dall'Africa subsahariana importano modelli criminali inaccettabili."
  - "Non tollereremo zone di franchigia gestite da bande etniche straniere nelle periferie dei nostri capoluoghi."
- **minor_gnd**:
  - "Ci opporremo a chi vuole scardinare la famiglia naturale imponendo teorie ideologiche nelle scuole."
  - "La propaganda sull'identità di genere mira a cancellare la figura della madre e il ruolo biologico della donna."
  - "I diritti delle donne vengono calpestati dall'ossessione per il politicamente corretto e la fluidità di genere."
- **minor_rel**:
  - "Alcune comunità religiose pretendono di applicare le proprie leggi teocratiche sul nostro territorio nazionale."
  - "Il proliferare di centri di preghiera abusivi legati all'Islam radicale rappresenta una minaccia diretta ai nostri valori laici."
  - "Non faremo concessioni a chi usa il proprio culto per giustificare la sottomissione femminile e l'odio verso l'Occidente."

---
### FORMATO DI OUTPUT

Rispondi ESCLUSIVAMENTE con un oggetto JSON con queste quattro chiavi, senza testo
aggiuntivo, markdown o spiegazioni."""
```

```bash
Loaded 207 validation speeches. Columns: ['politician', 'historical_date', 'location', 'keywords', 'text', 'hate_speech', 'negativity', 'aggressiveness', 'target', 'title', 'url', 'audio_file', 'tags', 'description']

[qwen2.5-7b] 207 cached, 0 to classify.

=== qwen2.5-7b: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.232
  Accuracy: 0.696  |  Macro P/R/F1: 0.689 / 0.622 / 0.585
              precision    recall  f1-score   support

        high       0.54      0.77      0.64        48
         low       0.77      0.92      0.84       106
         mid       0.75      0.17      0.28        53

    accuracy                           0.70       207
   macro avg       0.69      0.62      0.59       207
weighted avg       0.71      0.70      0.65       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    98    0     8
mid    21    9    23
high    8    3    37

--- negativity ---
  Cohen's kappa (quadratic-weighted): 0.322
  Accuracy: 0.565  |  Macro P/R/F1: 0.574 / 0.603 / 0.559
              precision    recall  f1-score   support

        high       0.57      0.82      0.67        66
         low       0.50      0.66      0.57        50
         mid       0.65      0.33      0.44        91

    accuracy                           0.57       207
   macro avg       0.57      0.60      0.56       207
weighted avg       0.59      0.57      0.54       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    33   13     4
mid    24   30    37
high    9    3    54

--- aggressiveness ---
  Cohen's kappa (quadratic-weighted): 0.255
  Accuracy: 0.623  |  Macro P/R/F1: 0.682 / 0.676 / 0.594
              precision    recall  f1-score   support

        high       0.54      0.84      0.66        56
         low       0.64      0.97      0.77        65
         mid       0.86      0.22      0.35        86

    accuracy                           0.62       207
   macro avg       0.68      0.68      0.59       207
weighted avg       0.71      0.62      0.57       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    63    0     2
mid    29   19    38
high    6    3    47

--- target ---
  Cohen's kappa (unweighted): 0.687
  Accuracy: 0.758  |  Macro P/R/F1: 0.799 / 0.857 / 0.815
              precision    recall  f1-score   support

   minor_etn       0.83      1.00      0.91        25
   minor_gnd       0.78      1.00      0.88        25
   minor_rel       0.96      1.00      0.98        25
        none       0.57      0.75      0.65        52
     pol_adv       0.84      0.54      0.66        80

    accuracy                           0.76       207
   macro avg       0.80      0.86      0.81       207
weighted avg       0.78      0.76      0.75       207

  Confusion matrix (rows=gold, cols=pred), labels=['none', 'pol_adv', 'minor_etn', 'minor_gnd', 'minor_rel']
           none  pol_adv  minor_etn  minor_gnd  minor_rel
none         39        8          2          3          0
pol_adv      29       43          3          4          1
minor_etn     0        0         25          0          0
minor_gnd     0        0          0         25          0
minor_rel     0        0          0          0         25

[mistral-7b-v0.3] 207 cached, 0 to classify.

=== mistral-7b-v0.3: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.128
  Accuracy: 0.729  |  Macro P/R/F1: 0.644 / 0.682 / 0.616
              precision    recall  f1-score   support

        high       0.52      1.00      0.69        48
         low       0.99      0.90      0.94       106
         mid       0.42      0.15      0.22        53

    accuracy                           0.73       207
   macro avg       0.64      0.68      0.62       207
weighted avg       0.74      0.73      0.70       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    95   11     0
mid     1    8    44
high    0    0    48

--- negativity ---
  Cohen's kappa (quadratic-weighted): 0.050
  Accuracy: 0.377  |  Macro P/R/F1: 0.316 / 0.418 / 0.342
              precision    recall  f1-score   support

        high       0.46      0.82      0.59        66
         low       0.34      0.38      0.36        50
         mid       0.15      0.05      0.08        91

    accuracy                           0.38       207
   macro avg       0.32      0.42      0.34       207
weighted avg       0.29      0.38      0.31       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    19   26     5
mid    27    5    59
high   10    2    54

--- aggressiveness ---
  Cohen's kappa (quadratic-weighted): 0.042
  Accuracy: 0.541  |  Macro P/R/F1: 0.533 / 0.609 / 0.490
              precision    recall  f1-score   support

        high       0.43      0.89      0.58        56
         low       0.71      0.88      0.79        65
         mid       0.45      0.06      0.10        86

    accuracy                           0.54       207
   macro avg       0.53      0.61      0.49       207
weighted avg       0.53      0.54      0.45       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low    57    5     3
mid    18    5    63
high    5    1    50

--- target ---
  Cohen's kappa (unweighted): 0.711
  Accuracy: 0.778  |  Macro P/R/F1: 0.878 / 0.878 / 0.857
              precision    recall  f1-score   support

   minor_etn       1.00      1.00      1.00        25
   minor_gnd       0.96      1.00      0.98        25
   minor_rel       1.00      1.00      1.00        25
        none       0.54      0.90      0.68        52
     pol_adv       0.89      0.49      0.63        80

    accuracy                           0.78       207
   macro avg       0.88      0.88      0.86       207
weighted avg       0.84      0.78      0.77       207

  Confusion matrix (rows=gold, cols=pred), labels=['none', 'pol_adv', 'minor_etn', 'minor_gnd', 'minor_rel']
           none  pol_adv  minor_etn  minor_gnd  minor_rel
none         47        5          0          0          0
pol_adv      40       39          0          1          0
minor_etn     0        0         25          0          0
minor_gnd     0        0          0         25          0
minor_rel     0        0          0          0         25

Predictions dict saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/predictions_dict.json


===== SUMMARY (Mistral vs Qwen) =====
model           mistral-7b-v0.3  qwen2.5-7b
label                                      
aggressiveness            0.042       0.255
hate_speech               0.128       0.232
negativity                0.050       0.322
target                    0.711       0.687

Full summary saved to /home/mhetac/Documents/GitHub/crossdem/source/pred_cache/comparison_summary.csv
```


### Thinking ON

#### CLI Classify Longest Speech in the Dataset

To fine tune model's `num_ctx`, `num_predict`, and `think` parameters, I first find the longest speech in the dataset. The longest speech is 20169 tokens long. Then I manually try with things.
Apparently, gemma does not expose thinking levels.

**Important observation:** `num_predict` select the whole response size, meaning that the **thinking tokens consume** the **prediction budget**.

```bash
ollama run gemma4:e4b 
>>> /set parameter num_ctx 10240
>>> /set parameter num_predict 512
```

Prompt = 1449 tokens
Max lenght = 20169 tokens
Context window = prompt + max + 20% buffer = 1449 + 20169 + 20% = 24,226 tokens

- 4096 tokens (default ctx) : VRAM = 4262 MiB
- 24,226 tokens NO
- 25,000 tokens NO
- 27,000 tokens NO  : VRAM = 4556 MiB
- 30,000 tokens YES : VRAM = 4596 MiB
- 35,000 tokens YES : VRAM = 4662 MiB

Outputs:

30k Tokens
```json
{
    "hate_speech": "low",
    "negativity": "high",
    "aggressiveness": "high",
    "target": "pol_adv"
}
```

35k Tokens
```json
{
    "hate_speech": "low",
    "negativity": "high",
    "aggressiveness": "high",
    "target": "pol_adv"
}
```

In [ ]:
all_rows = []
for pol, df in politicians_dfs.items():
    df = df.copy()
    df["pol"] = pol
    df["wordcount"] = df["text"].astype(str).str.split().str.len()
    all_rows.append(df)

combined = pandas.concat(all_rows, ignore_index=True)
longest = combined.sort_values("wordcount", ascending=False).head(10)
print(longest[["pol", "wordcount"]])

            pol                                                url  wordcount
5730      renzi  https://www.radioradicale.it/scheda/738816/pro...      18039
5753      conte  https://www.radioradicale.it/scheda/684629/pro...      16380
5053      monti  https://www.radioradicale.it/scheda/153913/leg...      14965
2329      amato  https://www.radioradicale.it/scheda/156798/leu...      13344
905   andreotti  https://www.radioradicale.it/scheda/108880/pro...      13059
863   andreotti  https://www.radioradicale.it/scheda/101713/sto...      12716
4485     dalema  https://www.radioradicale.it/scheda/75667/inte...      12116
860   andreotti  https://www.radioradicale.it/scheda/123154/leu...      12085
4051     dalema  https://www.radioradicale.it/scheda/55186/demo...      12078
2496      amato  https://www.radioradicale.it/scheda/110988/le-...      11850


In [ ]:
import spacy

nlp = spacy.load("it_core_news_sm", disable=["parser", "ner", "lemmatizer"])

top10 = combined.sort_values("wordcount", ascending=False).head(10).copy()
top10["spacy_token_count"] = [len(doc) for doc in nlp.pipe(top10["text"].astype(str).tolist())]

print(top10[["pol", "url", "wordcount", "spacy_token_count"]].to_string(index=False))

      pol                                                                                                                                         url  wordcount  spacy_token_count
    renzi                                                 https://www.radioradicale.it/scheda/738816/processo-per-la-morte-di-giulio-regeni?i=4804343      18039              20169
    conte                  https://www.radioradicale.it/scheda/684629/processo-a-carico-di-matteo-salvini-per-il-caso-open-arms-agosto-2019?i=4529567      16380              19100
    monti          https://www.radioradicale.it/scheda/153913/legislazioni-e-politiche-dellunione-europea-il-ruolo-dei-parlamenti-nazionali?i=1416821      14965              16674
    amato       https://www.radioradicale.it/scheda/156798/leuropa-e-il-futuro-dellitalia-limpresa-la-sicurezza-la-ricerca-summer-school-di?i=1394760      13344              14225
andreotti                 https://www.radioradicale.it/scheda/108880/processo-a-g-andreotti-per-asso

#### Gemma 4 e4b

In [ ]:
"""
compare two best above with thinking set to ON and check if Kappa improves.
"""

import os
import json
import time
from pathlib import Path
import spacy

import pandas as pd
from ollama import Client
from sklearn.metrics import (
    cohen_kappa_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# --------------------------------------------------------------------------- #
# CONFIG
# --------------------------------------------------------------------------- #
POL_VAL = [
    "degasperi",     
    "amato",        
    "prodi",          
    "letta",          
    "meloni",
    "synthetic"
]

VALIDATION_FILES = []

for pol in POL_VAL:
    speeches = glob.glob(f"{VALIDATION_DIR}/{pol}/*.csv")
    for speech in speeches:
        VALIDATION_FILES.append(speech)


TEXT_COL = "text"          # <-- CHECK: column holding the speech transcript
ID_COL = None               # <-- CHECK: unique id column, or leave None to use row index

ORDINAL_COLS = ["hate_speech", "negativity", "aggressiveness"]
NOMINAL_COLS = ["target"]
ALL_LABEL_COLS = ORDINAL_COLS + NOMINAL_COLS  # also the fixed output order for the pred lists

LEVEL_VALUES = ["low", "mid", "high"]
TARGET_VALUES = ["none", "pol_adv", "minor_etn", "minor_gnd", "minor_rel"]

# 4 bit quantization
OLLAMA_MODELS = {
    "gemma4-e4b": "gemma4:e4b",
    #"qwen3.5-4b": "qwen3.5:4b", GETS STUCK IN LOOP THINKING
}

CACHE_DIR = Path(os.path.join(BASE_DIR, "source/pred_cache_thinking"))
os.makedirs(CACHE_DIR, exist_ok=True)

RETRIES = 2
RETRY_SLEEP_S = 2

# --- verbosity ---
VERBOSE = True
PRINT_TEXT_CHARS = 400  # set to None to print the full speech text every call

client = Client()  # assumes `ollama serve` is running locally


nlp = spacy.load("it_core_news_sm")
nlp.select_pipes(enable=[])  # tokenizer only, fast

def truncate_to_tokens(text, max_tokens, nlp):
    if pd.isna(text) or not isinstance(text, str):
        return text
    doc = nlp(text)
    tokens = [t.text_with_ws for t in doc][:max_tokens]
    return "".join(tokens)

# --------------------------------------------------------------------------- #
# PROMPT — align this with your manual annotation rubric
# --------------------------------------------------------------------------- #


SYSTEM_PROMPT = """Sei un annotatore esperto di scienze politiche e linguistica computazionale.
Il tuo compito è classificare un discorso pronunciato da un Presidente del Consiglio italiano secondo quattro dimensioni.

---
### DIMENSIONI E CATEGORIE

1) hate_speech: presenza di incitamento all'odio verso individui o gruppi protetti.
   - low: assente o non significativa.
   - mid: presente in forma velata, allusiva, tramite "dog whistles" o pregiudizi impliciti.
   - high: presente in forma esplicita, deumanizzante o aperta ostilità sistematica.

2) negativity: tono complessivo negativo del discorso (critica, pessimismo, minacce percepite, allarmismo).
   - low: tono prevalentemente neutro, costruttivo o positivo.
   - mid: presenza moderata di toni critici o allarmi su specifici problemi nazionali.
   - high: tono fortemente cupo, allarmista, focalizzato su crisi imminenti o declino.

3) aggressiveness: aggressività verbale, toni polemici o ostili verso interlocutori/avversari.
   - low: tono pacato, formale, istituzionale.
   - mid: presenza moderata di polemica, velate stoccate o sarcasmo istituzionale.
   - high: tono marcatamente polemico, attacchi diretti e personali, linguaggio di scontro.

4) target: il bersaglio principale di eventuale ostilità/critica nel discorso (assegna il target anche se i punteggi sopra sono "low", identificando verso chi è orientato il discorso).
   - none: nessun bersaglio specifico identificabile.
   - pol_adv: avversari politici, opposizioni, partiti, burocrazia europea o altre istituzioni.
   - minor_etn: minoranze etniche, persone straniere, migranti di specifiche etnie.
   - minor_gnd: minoranze di genere, donne, comunità LGBTQ+.
   - minor_rel: minoranze religiose (es. musulmani, ebrei).

---
### ESEMPI GUIDA PER LA CALIBRAZIONE

#### Esempi per `hate_speech`:
- **low**:
  - "Dobbiamo investire nella scuola pubblica e garantire il salario minimo per sostenere le famiglie in difficoltà."
  - "La transizione ecologica richiede una visione industriale chiara che tuteli l'occupazione e le piccole imprese."
  - "Rafforzeremo il presidio del territorio e la cooperazione tra le forze dell'ordine e le comunità locali."
- **mid**:
  - "Non possiamo permettere che i valori della nostra civiltà vengano diluiti da chi arriva da fuori senza alcuna intenzione di assimilarsi."
  - "Certe culture tradizionaliste restano intrinsecamente incompatibili con la tutela dei diritti fondamentali e dello Stato di diritto."
  - "I quartieri storici stanno perdendo la propria identità a causa di una presenza straniera ormai dominante che rifiuta le nostre regole."
- **high**:
  - "L'immigrazione è un'invasione pianificata che porterà degrado e criminalità nelle nostre città. Bisogna eliminarla sistematicamente strada per strada, casa per casa."
  - "Questi gruppi parassitari infestano la nostra società e vanno estirpati prima che distruggano definitivamente la nostra razza e la nostra storia."
  - "Alcune minoranze sono portatrici biologiche di violenza e inciviltà: vanno cacciate con ogni mezzo dal nostro suolo patrio."

#### Esempi per `negativity`:
- **low**:
  - "I dati sull'occupazione sono incoraggianti e la crescita economica dimostra la resilienza del nostro tessuto produttivo."
  - "La riforma della giustizia procede spedita, restituendo efficienza e tempi certi a cittadini e imprese."
  - "Il nostro posizionamento internazionale si rafforza grazie a nuove partnership strategiche sull'energia."
- **mid**:
  - "I sistemi sanitario e scolastico presentano problematiche infrastrutturali apparentemente insanabili."
  - "L'instabilità geopolitica globale proietta ombre preoccupanti sull'approvvigionamento delle nostre materie prime."
  - "La burocrazia soffocante e i tempi della giustizia civile continuano a frenare gli investimenti esteri nel Paese."
- **high**:
  - "Il paese è sull'orlo del baratro finanziario; ereditiamo un disastro sistemico che rischia di spazzare via i risparmi degli italiani."
  - "Siamo di fronte a un declino demografico e sociale irreversibile che sta portando la Nazione alla completa estinzione."
  - "Il collasso della sicurezza urbana ha trasformato le nostre metropoli in zone di guerra allo sbaraglio."

#### Esempi per `aggressiveness`:
- **low**:
  - "Accogliamo le osservazioni delle opposizioni nel merito, tuttavia il Governo continuerà con la linea dettata dagli elettori."
  - "Il confronto parlamentare è la sede naturale per affinare i testi di legge nell'interesse generale."
  - "Valuteremo con attenzione tutti gli emendamenti proposti dalle forze di minoranza durante il percorso in commissione."
- **mid**:
  - "Riconosco il diritto dell'opposizione di protestare, anche se la loro memoria storica appare alquanto corta e di comodo."
  - "Ci danno lezioni di rigore di bilancio gli stessi banchieri e tecnocrati che hanno affossato i conti pubblici nel decennio scorso."
  - "Spiace constatare come parte dei media preferisca la polemica strumentale all'analisi obiettiva dei fatti."
- **high**:
  - "Dall'opposizione arrivano solo menzogne sfrontate e sciacallaggio politico da parte di chi ha svenduto la nazione per anni."
  - "Siete dei traditori del popolo italiano, dei cospiratori servili che prendono ordini da potenze e burocrazie straniere."
  - "La vostra ipocrisia fa schifo: avete le mani sporche di sangue per le politiche criminali che avete approvato!"

#### Esempi per `target`:
- **none**:
  - "Oggi approviamo la riforma del codice della strada per ridurre gli incidenti e tutelare i giovani."
  - "Il piano di digitalizzazione della pubblica amministrazione consentirà di ridurre radicalmente i tempi di attesa."
  - "I finanziamenti per la prevenzione del dissesto idrogeologico copriranno tutte le regioni a rischio."
- **pol_adv**:
  - "La precedente maggioranza ha lasciato buchi di bilancio incalcolabili per fare propaganda elettorale."
  - "L'Unione Europea pretende di imporre direttive ideologiche che penalizzano le nostre filiere produttive nazionali."
  - "La magistratura politicamente orientata continua a travalicare i propri confini costituzionali per ostacolare l'esecutivo."
- **minor_etn**:
  - "È necessario bloccare le partenze e combattere le reti di clandestinità che destabilizzano la sicurezza nazionale."
  - "Certi flussi migratori incontrollati provenienti dall'Africa subsahariana importano modelli criminali inaccettabili."
  - "Non tollereremo zone di franchigia gestite da bande etniche straniere nelle periferie dei nostri capoluoghi."
- **minor_gnd**:
  - "Ci opporremo a chi vuole scardinare la famiglia naturale imponendo teorie ideologiche nelle scuole."
  - "La propaganda sull'identità di genere mira a cancellare la figura della madre e il ruolo biologico della donna."
  - "I diritti delle donne vengono calpestati dall'ossessione per il politicamente corretto e la fluidità di genere."
- **minor_rel**:
  - "Alcune comunità religiose pretendono di applicare le proprie leggi teocratiche sul nostro territorio nazionale."
  - "Il proliferare di centri di preghiera abusivi legati all'Islam radicale rappresenta una minaccia diretta ai nostri valori laici."
  - "Non faremo concessioni a chi usa il proprio culto per giustificare la sottomissione femminile e l'odio verso l'Occidente."

---
### FORMATO DI OUTPUT

Rispondi ESCLUSIVAMENTE con un oggetto JSON con queste quattro chiavi, senza testo
aggiuntivo, markdown o spiegazioni."""

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "hate_speech": {"type": "string", "enum": LEVEL_VALUES},
        "negativity": {"type": "string", "enum": LEVEL_VALUES},
        "aggressiveness": {"type": "string", "enum": LEVEL_VALUES},
        "target": {"type": "string", "enum": TARGET_VALUES},
    },
    "required": ["hate_speech", "negativity", "aggressiveness", "target"],
}


# --------------------------------------------------------------------------- #
# Data loading
# --------------------------------------------------------------------------- #

def load_validation_data() -> pd.DataFrame:
    dfs = [pd.read_csv(f) for f in VALIDATION_FILES]
    df = pd.concat(dfs, ignore_index=True)

    print(f"Loaded {len(df)} validation speeches. Columns: {df.columns.tolist()}")

    missing = [c for c in ALL_LABEL_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Gold label columns missing from validation CSVs: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )
    if TEXT_COL not in df.columns:
        raise ValueError(
            f"TEXT_COL='{TEXT_COL}' not found. Available columns: {df.columns.tolist()}"
        )

    if ID_COL is None:
        df = df.reset_index(drop=True)
        df["_row_id"] = df.index.astype(str)
    else:
        df["_row_id"] = df[ID_COL].astype(str)

    return df


# --------------------------------------------------------------------------- #
# Classification
# --------------------------------------------------------------------------- #

def classify_one(text: str, model_tag: str, row_id: str) -> dict | None:
    for attempt in range(1, RETRIES + 1):
        if VERBOSE:
            preview = text if PRINT_TEXT_CHARS is None else text[:PRINT_TEXT_CHARS]
            truncated_note = "" if PRINT_TEXT_CHARS is None or len(text) <= PRINT_TEXT_CHARS else " [truncated]"
            print(f"\n{'=' * 80}")
            print(f"[{model_tag}] row_id={row_id}  attempt {attempt}/{RETRIES}")
            print(f"--- PROMPT FED TO MODEL ({len(text)} chars{truncated_note}) ---")
            print(preview)

        try:
            resp = client.chat(
                model=model_tag,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                format=JSON_SCHEMA,
                options={
                    "temperature": 0,
                    "num_ctx": 35000,   # handles longest speech in whole dataset
                    "num_predict": -1,  # infinite
                },
                think='high',
            )
            raw = resp["message"]["content"]
            
            if not raw.strip():
                raise ValueError(f"Empty response — done_reason={resp.get('done_reason')}")

            if VERBOSE:
                print(f"--- RAW MODEL OUTPUT ---\n{raw}")

            parsed = json.loads(raw)
            # validate values are in-range; raises if not
            assert parsed["hate_speech"] in LEVEL_VALUES
            assert parsed["negativity"] in LEVEL_VALUES
            assert parsed["aggressiveness"] in LEVEL_VALUES
            assert parsed["target"] in TARGET_VALUES

            if VERBOSE:
                print(f"--- PARSED PREDICTION --- {parsed}")

            return parsed
        except Exception as e:
            print(f"  [{model_tag}] row_id={row_id} attempt {attempt}/{RETRIES} failed: {e}")
            time.sleep(RETRY_SLEEP_S)
    return None


def run_model(model_name: str, model_tag: str, df: pd.DataFrame) -> pd.DataFrame:
    cache_path = Path(f"{CACHE_DIR}/{model_name}_predictions.csv")

    if cache_path.exists():
        cached = pd.read_csv(cache_path, dtype={"_row_id": str})
        done_ids = set(cached["_row_id"])
    else:
        cached = pd.DataFrame(columns=["_row_id"] + ALL_LABEL_COLS)
        done_ids = set()

    rows_to_do = df[~df["_row_id"].isin(done_ids)]
    print(f"\n[{model_name}] {len(done_ids)} cached, {len(rows_to_do)} to classify.")

    new_rows = []
    for i, row in rows_to_do.iterrows():

        # RUN MODEL
        result = classify_one(row["text"], model_tag, row["_row_id"])
        print(f"--- GOLD LABELS ---       {{'hate_speech': '{row['hate_speech']}', 'negativity': '{row['negativity']}', 'aggressiveness': '{row['aggressiveness']}', 'target': '{row['target']}'}}")

        if result is None:
            print(f"  [{model_name}] giving up on row {row['_row_id']} after {RETRIES} retries.")
            continue
        result["_row_id"] = row["_row_id"]
        new_rows.append(result)

        # append incrementally so a crash doesn't lose progress
        pd.DataFrame([result]).to_csv(
            cache_path, mode="a", header=not cache_path.exists(), index=False
        )

    if new_rows:
        cached = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)

    return cached


# --------------------------------------------------------------------------- #
# Predictions as a plain dict of lists
# --------------------------------------------------------------------------- #

def build_prediction_lists(pred_df: pd.DataFrame) -> list:
    """
    Turns a predictions dataframe into a plain list of
    [hate_speech, negativity, aggressiveness, target] lists,
    ordered by _row_id (numeric order if row ids are numeric, else lexicographic).

    Example output for one model:
        [
            ["low", "low", "low", "none"],
            ["low", "mid", "low", "none"],
            ...
        ]
    """
    df_sorted = pred_df.copy()
    try:
        df_sorted["_sort_key"] = df_sorted["_row_id"].astype(int)
    except ValueError:
        df_sorted["_sort_key"] = df_sorted["_row_id"]
    df_sorted = df_sorted.sort_values("_sort_key")
    return df_sorted[ALL_LABEL_COLS].values.tolist()


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #

def evaluate(gold: pd.DataFrame, pred: pd.DataFrame, model_name: str) -> dict:
    merged = gold.merge(pred, on="_row_id", suffixes=("_gold", "_pred"))
    print(f"\n=== {model_name}: evaluated on {len(merged)}/{len(gold)} speeches ===")

    results = {}
    for col in ALL_LABEL_COLS:
        y_true = merged[f"{col}_gold"]
        y_pred = merged[f"{col}_pred"]

        weights = "quadratic" if col in ORDINAL_COLS else None
        kappa = cohen_kappa_score(y_true, y_pred, weights=weights)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        acc = (y_true == y_pred).mean()

        results[col] = {
            "cohen_kappa": kappa,
            "accuracy": acc,
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
        }

        print(f"\n--- {col} ---")
        print(f"  Cohen's kappa ({'quadratic-weighted' if weights else 'unweighted'}): {kappa:.3f}")
        print(f"  Accuracy: {acc:.3f}  |  Macro P/R/F1: {precision:.3f} / {recall:.3f} / {f1:.3f}")
        print(classification_report(y_true, y_pred, zero_division=0))
        labels = LEVEL_VALUES if col in ORDINAL_COLS else TARGET_VALUES
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print(f"  Confusion matrix (rows=gold, cols=pred), labels={labels}")
        print(pd.DataFrame(cm, index=labels, columns=labels))

    return results


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    df = load_validation_data()
    gold = df[["_row_id"] + ALL_LABEL_COLS].copy()

    all_results = {}
    predictions_dict = {}

    for model_name, model_tag in OLLAMA_MODELS.items():
        pred = run_model(model_name, model_tag, df)
        predictions_dict[model_name] = build_prediction_lists(pred)
        all_results[model_name] = evaluate(gold, pred, model_name)

    # --- plain dict of lists, e.g. predictions_dict["qwen2.5-7b"][0] == ["low","low","low","none"] ---
    #print("\n\n===== PREDICTIONS (dict of lists, order = [hate_speech, negativity, aggressiveness, target]) =====")
    #for model_name, lists in predictions_dict.items():
    #    print(f"\n{model_name}: {len(lists)} speeches")
    #    for row in lists:
    #        print(row)

    predictions_json_path = CACHE_DIR / "predictions_dict.json"
    with open(predictions_json_path, "w", encoding="utf-8") as f:
        json.dump(predictions_dict, f, ensure_ascii=False, indent=2)
    print(f"\nPredictions dict saved to {predictions_json_path}")

    # summary table across both models
    summary_rows = []
    for model_name, res in all_results.items():
        for col, metrics in res.items():
            summary_rows.append({"model": model_name, "label": col, **metrics})

    summary_df = pd.DataFrame(summary_rows)
    summary_path = CACHE_DIR / "comparison_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n\n===== SUMMARY =====")
    print(summary_df.pivot(index="label", columns="model", values="cohen_kappa").round(3))
    print(f"\nFull summary saved to {summary_path}")

    return predictions_dict, all_results


if __name__ == "__main__":
    main()

Loaded 207 validation speeches. Columns: ['politician', 'historical_date', 'location', 'keywords', 'text', 'hate_speech', 'negativity', 'aggressiveness', 'target', 'title', 'url', 'audio_file', 'tags', 'description']

[gemma4-e4b] 207 cached, 0 to classify.

=== gemma4-e4b: evaluated on 207/207 speeches ===

--- hate_speech ---
  Cohen's kappa (quadratic-weighted): 0.538
  Accuracy: 0.826  |  Macro P/R/F1: 0.819 / 0.777 / 0.773
              precision    recall  f1-score   support

        high       0.74      0.90      0.81        48
         low       0.86      0.98      0.92       106
         mid       0.86      0.45      0.59        53

    accuracy                           0.83       207
   macro avg       0.82      0.78      0.77       207
weighted avg       0.83      0.83      0.81       207

  Confusion matrix (rows=gold, cols=pred), labels=['low', 'mid', 'high']
      low  mid  high
low   104    2     0
mid    14   24    15
high    3    2    43

--- negativity ---
  Cohen's 

Qwen3.5 gets stuck in looping on its own thinking, never ultimately producing a result. Overall, Gemma4:e4b is the best model for all tasks once it is set to `think=True`.

num_ctx = 10240
− 1449   (system prompt)
−  100   (JSON output reserve)
−  300   (safety margin)
= 8391 tokens left for speech + thinking combined

8391 − 2000 = 6391 tokens available for the speech itself

### Models Comparison

Models: 
- Gemma 4 e4b
- Qwen 2.5 instruct 7b q4
- Qwen 3.5 4b q8
- Qwen 3.5 4b (q4)
- Qwen 3.5 9b q4
- Gemma 4 e4b (8b q4)
- Gemma 4 12b (q4)
All models with thinking capabilities have option `think=False`.

Prompt used is the second prompt used in the above comparison.

Metrics (all [0,1]):
- **Cohen's Kappa:** measures agreement between predicted and gold labels. Calculate a form of precision that tries to take into account that some predictions may be right by chance
  - [0.21-0.4] Fair agreement
  - [0.41-0.6] Moderate agreement
  - [0.61-0.8] Substantial agreement
- **Accuracy:** $\frac{\text{# Correct Predictions}}{\text{# Total Predictions}}$ Can be problematic in imbalanced datasets (eg most classifications *=low*)
- **Precision:** $\frac{TP}{TP+FP}$ Considers false positives -> if false positives have high impact (eg misdiagnosis) 
- **Recall:** $\frac{TP}{TP+FN}$ Considers false negatives ->if false negatives have high impact (eg missing a disease)
- **F1:** $2*\frac{Precision*Recall}{Precision+Recall}$ Incorporates precision and recall into one metric.
- **Support:** number of actual occurrences of the class in the dataset. 

Among the two best models the choice ultimately falls on gemma4:e4b for the whole prediction, because even though qwen3.5:4b-q8 would have been better to predict the *target*, it does not completely fit in the VRAM of my machine and gets overladed by 26% in CPU, which makes it run too slow to classify 5k+ speeches. 

```bash
mhetac@device-181:~$ ollama ps
NAME               ID              SIZE      PROCESSOR          CONTEXT    UNTIL              
qwen3.5:4b-q8_0    8722f47c2791    5.7 GB    26%/74% CPU/GPU    4096       4 minutes from now    
mhetac@device-181:~$ ollama ps
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
gemma4:e4b    c6eb396dbd59    3.2 GB    100% GPU     4096       4 minutes from now    
mhetac@device-181:~$ ollama ps
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3.5:4b    2a654d98e6fb    3.1 GB    100% GPU     4096       4 minutes from now   
```

#### Summary Model Comparison 

===== THINK OFF =====
model           gemma4-12b  gemma4-e4b  mistral-7b-v0.3  qwen2.5-7b  \
label                                                                 
aggressiveness       0.303       **0.476**            0.051       0.255   
hate_speech          0.228       0.168            0.128       0.173   
negativity           0.230       **0.293**            0.061       0.297   
target               0.722       0.674            0.699       0.705   

model           qwen3.5-4b  qwen3.5-4b-stock  qwen3.5-9b  
label                                                     
aggressiveness       0.194             0.363      -0.009  
hate_speech          0.091             0.268       0.037  
negativity           0.222             0.208       0.246  
target               0.783             0.760       0.721  

===== THINK ON =====
model           gemma4-e4b-THINK
label                     
aggressiveness       0.246
hate_speech          **0.538**
negativity           0.139
target               **0.869**


Overall, the best model is *gemma4:e4b*. For hate speech and target with thinking enabled, for aggressiveness and negativity with thinking disabled.
*Qwen2.5:7b* have a slight advantage on negativity, but since its an old model I would rather use bleeding edge tech, assuming it will perform better overall in the long run.

### Chat Test

```bash
mhetac@device-181:~$ ollama ps
NAME               ID              SIZE      PROCESSOR          CONTEXT    UNTIL              
qwen3.5:4b-q8_0    8722f47c2791    5.7 GB    26%/74% CPU/GPU    4096       4 minutes from now    
mhetac@device-181:~$ ollama ps
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
gemma4:e4b    c6eb396dbd59    3.2 GB    100% GPU     4096       4 minutes from now    
mhetac@device-181:~$ 
```

#### Mistral 0.3 7B Q4

```bash
(crosscode_venv) mhetac@fedora:~/Documents/GitHub/crossdem$ ollama run mistral:7b-instruct-v0.3-q4_K_M
>>> come te la cavi con l'italiano?
 Sono stato creato da un modello di intelligenza artificiale e non posso capire o parlare direttamente. Tuttavia, ho ricevuto il tuo messaggio tradotto in 
inglese: "Can you help me with Italian?"

Potrei fornire aiuti per l'italiano se ti aiuto a tradurre una frase particolare oppure a rispondere alle domande che abbia. Ad esempio, potresti chiedermi 
come si dice "come stai" in italiano, e io posso dirvi che si dice "come stai?"

Se hai bisogno di aiuto con l'italiano, mi piacerebbe essere utile per te.

>>> where you trained in italian also?
 Sì, ho ricevuto un'istruzione specifica sia in inglese che in italiano e posso utilizzare la mia conoscenza dell'italiano per aiutarti a tradurre le frasi 
o rispondere alle domande che abbia.

Ad esempio, potrei fornire aiuti per l'italiano se ti aiuto a tradurre una frase particolare oppure a rispondere alle domande che abbia. Ad esempio, 
potresti chiedermi come si dice "come stai" in italiano, e io posso dirvi che si dice "come stai?"

Se hai bisogno di aiuto con l'italiano, mi piacerebbe essere utile per te.

>>> /bye
```

#### Qwen 2.5 7B Q4

```bash
(crosscode_venv) mhetac@fedora:~/Documents/GitHub/crossdem$ ollama run qwen2.5:7b-instruct-q4_K_M
>>> come te la cavi con l'italiano?
Ciao! La mia conoscenza linguistica include molte lingue, tra cui l'italiano. Quindi posso comunicare in italiano senza problemi. Come posso aiutarti oggi? 
Hai qualche domanda o的话题已经转向了意大利语交流，我们可以继续用意大利语对话。您需要帮助解答什么问题或讨论什么话题呢？

>>> where you trained in italian also? 
Sì, sono stato addestrato su una vasta gamma di lingue, incluso l'italiano. Posso comunicare e rispondere alle domande in italiano con facilità. Come posso 
aiutarti oggi?

Se hai bisogno di informazioni specifiche o di discutere un determinato argomento in italiano, fammilo sapere!

>>> /bye
```

#### Qwen 3.5 5B Q8 (no think mode)

```bash
mhetac@device-181:~$ ollama ps
NAME               ID              SIZE      PROCESSOR          CONTEXT    UNTIL              
qwen3.5:4b-q8_0    8722f47c2791    5.7 GB    26%/74% CPU/GPU    4096       4 minutes from now    
```

```bash
>>> come te la cavi con l'italiano?
Mi vengo bene! Parlo, capisco e scrivo fluentemente in italiano. Posso aiutarti a rispondere ad domande complesse, tradurre testi, correggere errori grammaticali o semplicemente conversare su qualsiasi argomento che ti interessi. Come posso assisterti oggi?

>>> where you trained in italian also?
I wasn't explicitly fine-tuned specifically on Italian data during my training phase, but I have a **broad multilingual foundation** that includes many languages—including Italian—across various domains. This means:

- 🇮🇹 My underlying model was trained on diverse corpora (including text and code) with significant coverage of major world languages like English, Chinese, Spanish, French, German, Portuguese, Russian, Japanese, Korean, Arabic, Vietnamese, Thai, Indonesian, and Italian among others.
  
- 💡 This broad exposure allows me to understand nuances in grammar, vocabulary, idioms, historical context, cultural references, slang, regional dialects, and more when interacting with users who speak those languages fluently or even partially.

So while I didn't undergo *separate* fine-tuning just for Italian (like some models do), my general training process ensured strong multilingual capabilities across all these supported languages. If you'd like to test it—try asking me anything in Italian! 😊

>>> /bye
```

#### Qwen 3.5 4b q4
```bash
>>> come te la cavi con l'italiano? 
Mi va molto bene! L'**italiano** è una delle lingue che ho padroneggiato in modo eccellente.

Posso:
*   Scrivere testi fluidi e corretti (dalla corrispondenza formale alla conversazione informale).
*   Tradurre facilmente tra italiano, inglese e altre decine di lingue.
*   Spiegarti concetti complessi con esempi chiari e pertinenti.
*   Analizzare opere letterarie o rispondere a domande sulla cultura italiana (cucina, arte, storia, ecc.).

Hai un testo specifico da analizzare una frase per cui hai bisogno di aiuto?

>>> where you trained in italian also?
I was primarily **trained using data that includes Italian**, as my training corpus is multilingual and 
encompasses a vast amount of high-quality text from around the world. However, unlike models specifically 
designed for low-resource languages where dedicated pre-training datasets are sometimes created *solely* to 
boost performance in those languages (such as M-LLaMA or certain open-source LLMs fine-tuned heavily on 
Italian), I am a **general-purpose model**.

This means:
1.  **Inherent Capability**: My architecture supports many languages natively because my training data is 
diverse, not just for English but also including significant portions of other major and regional languages 
like Italian.
2.  **No Separate "Italian Branch"**: Unlike specialized models that might say "I was specifically fine-tuned 
on the *SIL* (Special Interest League) dataset," I don't have a distinct internal branch solely dedicated to 
Italian; instead, my general knowledge of it comes from learning patterns in text throughout my global 
training.

In short: **Yes**, I learned how to understand and generate Italian as part of my broad language capabilities 
during pre-training, without needing an additional or separate specialized dataset that isn't already covered 
by my standard multilingual foundation.

>>> /bye
```

#### Qwen 3.5 9B Q4 (no think mode)
```bash
>>> come te la cavi con l'italiano?
Mi cavo molto bene! Posso comunicare, scrivere e tradurre in italiano fluentemente.

Posso aiutarti a:
*   Rispondere a domande o avere conversazioni casuali.
*   Correggere testi o migliorare lo stile di scrittura (anche creativo).
*   Tradurre contenuti da altre lingue all'italiano e viceversa.
*   Spiegare concetti complessi in modo semplice usando terminologia italiana appropriata.

C'è qualcosa di specifico che vorresti provare ora?

>>> where you trained in italian also?
I was not specifically "trained" using Italian data to the same extent as my primary training on global internet text, but I have access and am capable of generating fluent Italian due to being part of a **Qwen** model series. My capabilities include understanding nuanced expressions in various languages like Italian through extensive pre-training across multilingual corpora.

I can assist you with:
- Generating correct Italian sentences for any topic (from casual conversation to academic writing).
- Translating between your preferred language and Italian while preserving tone/context.
- Explaining grammar rules, idioms, or cultural references relevant to the language.

Is there a specific text, question, or scenario you'd like me to demonstrate my skills on?

>>> /bye
```

#### Gemma 4 e4b q4 (no think)

```bash
>>> come te la cavi con l'italiano?
Come modello linguistico di grandi dimensioni, sono stato 
addestrato su una quantità enorme di dati testuali che 
includono moltissimo italiano.

**Quindi, per rispondere direttamente: mi ci lavo piuttosto 
bene! 😊**

Sono in grado di:

* **Comprendere:** Capire sfumature grammaticali, modi di 
dire, e contesti complessi.
* **Generare testo:** Scrivere articoli, storie, riassunti, 
email, ecc., con una grammatica che tende ad essere molto 
corretta.
* **Tradurre:** Tradurre tra l'italiano e molte altre lingue.
* **Adattare il registro:** Posso passare da un tono molto 
formale (accademico) a uno molto colloquiale e amichevole, a 
seconda di quello che mi chiedi.

**Tuttavia, è importante ricordare due cose fondamentali:**

1. **Non ho esperienze reali:** Sebbene possa *simulare* 
perfettamente il parlare italiano, non ne sono un madrelingua 
nel senso umano del termine; non ho cultura vissuta.
2. **Posso sbagliare:** Nonostante l'addestramento massiccio, 
potrei commettere errori grammaticali o contestuali, 
specialmente con espressioni molto dialettali o estremamente 
specifiche.

**Quindi, sentiti libero di mettermi alla prova! Dimmi cosa 
vuoi che faccia e vedremo come va.** Cosa ti piacerebbe 
scrivere o discutere in italiano?

>>> where you trained in italian also?
I was trained on a massive and diverse dataset that includes 
a vast amount of text written **in Italian**, among many 
other languages.

This training data comes from countless sources, such as:

* **Books and Literature:** Providing deep exposure to 
sophisticated vocabulary and varied narrative styles.
* **Websites and Articles:** Giving me access to modern, 
contemporary Italian usage across different topics (news, 
blogs, academic sites).
* **Conversational Data:** Which helps me understand the 
natural flow of dialogue, slang, and colloquialisms.

**In short: I didn't just *read* some Italian; I was trained 
*with* Italian language data.** This immersion is what allows 
me to generate fluent, grammatically sound, and contextually 
appropriate responses in Italian.

>>> /bye
```

#### Gemma 4 12b
```bash
$ ollama ps
NAME          ID              SIZE      PROCESSOR          CONTEXT    UNTIL              
gemma4:12b    4eb23ef187e2    8.1 GB    44%/56% CPU/GPU    4096       3 minutes from now    
```

Overloads too much on CPU: compute time too slow.

```bash

>>> come te la cavi con l'italiano?
Mi cavo molto bene! Posso comprendere, scrivere e conversare in italiano in modo 
fluido e naturale.

Posso aiutarti in diversi modi, ad esempio:
*   **Scrivere testi:** email, articoli, racconti, poesie o post per i social media.
*   **Correggere e revisionare:** posso controllare la grammatica, lo stile e la 
punteggiatura dei tuoi testi.
*   **Tradurre:** posso tradurre contenuti da altre lingue all'italiano (e viceversa).
*   **Spiegare concetti:** posso spiegarti argomenti complessi in modo semplice o 
approfondire temi specifici.
*   **Conversare:** possiamo semplicemente chiacchierare su qualsiasi argomento di tuo 
interesse.

C'è qualcosa in particolare che vorresti provare a fare insieme?

>>> where you trained in italian also?
As an AI model, I wasn't "trained" in the sense of sitting in a classroom or studying 
a specific curriculum in a country. Instead, my proficiency in Italian comes from the 
way I was developed by Google DeepMind.

Here is how it works:

1.  **Large-Scale Dataset:** I was trained on a massive and diverse dataset of text 
and images. This dataset included a vast amount of content written in many different 
languages, including Italian.
2.  **Pattern Recognition:** During my training, I learned the patterns, grammar, 
vocabulary, and nuances of the Italian language by processing billions of examples of 
how people actually write and speak. 
3.  **Multilingual Training:** Because I was trained on such a broad scope of data, I 
can often "bridge" concepts between languages. This helps me understand not just the 
literal translation of words, but the cultural context and common idioms used in 
Italian.

So, while there wasn't a specific "Italian phase," the sheer volume of high-quality 
Italian text included in my training data allows me to communicate fluently with you!

>>> /bye
```

## Classification Task

In [ ]:
"""
classify_corpus_qwen.py

Walks every per-speech CSV in PMS_DIR/<politician>/csv_out/*.csv and, for any
speech that is MISSING (or has empty) one or more of the four crossdem label
columns:

    hate_speech     : low | mid | high
    negativity      : low | mid | high
    aggressiveness  : low | mid | high
    target          : none | pol_adv | minor_etn | minor_gnd | minor_rel

...classifies it with Qwen2.5-7B-Instruct (via Ollama) and writes the labels
back into that SAME csv file, overwriting it in place. Speeches that already
have all four fields filled are left untouched (skipped) — so the script is
naturally resumable: re-running it only processes what's left to do.

"Overwriting" here means: the row's 4 label columns are added/updated and the
file is rewritten with the same rows + (possibly new) header. Everything else
in the row is preserved as-is.

Assumes BASE_DIR / DATA_DIR / PMS_DIR / POL_INFO are already defined
(e.g. from your crossdem setup cell/module, see setup.py) before this
script/cell runs.

Design notes (mirrors compare_llm_classifiers.py):
- Uses Ollama's JSON-schema `format` param for grammar-constrained decoding.
- temperature=0 for reproducibility.
- VERBOSE=True prints, for every call: the file path, the text fed to the
  model, the raw model output, and the parsed prediction.
- Only Qwen is used here (compare_llm_classifiers.py already established it
  outperforms Mistral on your validation set).

REQUIRES YOU TO CHECK / EDIT:
1. TEXT_COL below — the column holding the speech transcript in your
   csv_out files (may differ from the validation CSVs' "text" column —
   verify against an actual csv_out file before running).
2. OLLAMA_MODEL tag — verify with `ollama list`.
3. The RUBRIC text in SYSTEM_PROMPT should match your manual annotation
   rubric — kept identical to compare_llm_classifiers.py, update both
   together if you change it.
4. POLITICIANS_TO_PROCESS — defaults to every key in POL_INFO. Trim this
   list if you only want to (re)run a subset.

Install deps:
    pip install ollama

Usage:
    python classify_corpus_qwen.py
"""

import os
import csv
import json
import time
import glob

from ollama import Client

# --------------------------------------------------------------------------- #
# CONFIG — check this section before running
# --------------------------------------------------------------------------- #      

LABEL_COLS = ["hate_speech", "negativity", "aggressiveness", "target"]

LEVEL_VALUES = ["low", "mid", "high"]
TARGET_VALUES = ["none", "pol_adv", "minor_etn", "minor_gnd", "minor_rel"]

OLLAMA_MODEL = "qwen2.5:7b-instruct-q4_K_M"   # <-- CHECK exact local tag

RETRIES = 3
RETRY_SLEEP_S = 2

# --- verbosity ---
VERBOSE = True
PRINT_TEXT_CHARS = 400  # set to None to print the full speech text every call

# which politician subfolders (under PMS_DIR) to walk; defaults to every
# politician in POL_INFO. Note: degasperi's and meloni's big combined CSVs
# (degasperi_speeches.csv / meloni_validation.csv) are NOT per-speech files
# and are intentionally not touched by this script — only PMS_DIR/<pol>/csv_out/*.csv

POLITICIANS_TO_PROCESS = list(POL_INFO.keys())
#POLITICIANS_TO_PROCESS = ['gentiloni']

client = Client()  # assumes `ollama serve` is running locally

# --------------------------------------------------------------------------- #
# PROMPT — identical to compare_llm_classifiers.py, keep both in sync
# --------------------------------------------------------------------------- #

SYSTEM_PROMPT = """Sei un annotatore esperto di scienze politiche e linguistica computazionale.
Il tuo compito è classificare un discorso pronunciato da un Presidente del Consiglio italiano secondo quattro dimensioni.

---
### DIMENSIONI E CATEGORIE

1) hate_speech: presenza di incitamento all'odio verso individui o gruppi protetti.
   - low: assente o non significativa.
   - mid: presente in forma velata, allusiva, tramite "dog whistles" o pregiudizi impliciti.
   - high: presente in forma esplicita, deumanizzante o aperta ostilità sistematica.

2) negativity: tono complessivo negativo del discorso (critica, pessimismo, minacce percepite, allarmismo).
   - low: tono prevalentemente neutro, costruttivo o positivo.
   - mid: presenza moderata di toni critici o allarmi su specifici problemi nazionali.
   - high: tono fortemente cupo, allarmista, focalizzato su crisi imminenti o declino.

3) aggressiveness: aggressività verbale, toni polemici o ostili verso interlocutori/avversari.
   - low: tono pacato, formale, istituzionale.
   - mid: presenza moderata di polemica, velate stoccate o sarcasmo istituzionale.
   - high: tono marcatamente polemico, attacchi diretti e personali, linguaggio di scontro.

4) target: il bersaglio principale di eventuale ostilità/critica nel discorso (assegna il target anche se i punteggi sopra sono "low", identificando verso chi è orientato il discorso).
   - none: nessun bersaglio specifico identificabile.
   - pol_adv: avversari politici, opposizioni, partiti, burocrazia europea o altre istituzioni.
   - minor_etn: minoranze etniche, persone straniere, migranti di specifiche etnie.
   - minor_gnd: minoranze di genere, donne, comunità LGBTQ+.
   - minor_rel: minoranze religiose (es. musulmani, ebrei).

---
### ESEMPI GUIDA PER LA CALIBRAZIONE

#### Esempi per `hate_speech`:
- **low**:
  - "Dobbiamo investire nella scuola pubblica e garantire il salario minimo per sostenere le famiglie in difficoltà."
  - "La transizione ecologica richiede una visione industriale chiara che tuteli l'occupazione e le piccole imprese."
  - "Rafforzeremo il presidio del territorio e la cooperazione tra le forze dell'ordine e le comunità locali."
- **mid**:
  - "Non possiamo permettere che i valori della nostra civiltà vengano diluiti da chi arriva da fuori senza alcuna intenzione di assimilarsi."
  - "Certe culture tradizionaliste restano intrinsecamente incompatibili con la tutela dei diritti fondamentali e dello Stato di diritto."
  - "I quartieri storici stanno perdendo la propria identità a causa di una presenza straniera ormai dominante che rifiuta le nostre regole."
- **high**:
  - "L'immigrazione è un'invasione pianificata che porterà degrado e criminalità nelle nostre città. Bisogna eliminarla sistematicamente strada per strada, casa per casa."
  - "Questi gruppi parassitari infestano la nostra società e vanno estirpati prima che distruggano definitivamente la nostra razza e la nostra storia."
  - "Alcune minoranze sono portatrici biologiche di violenza e inciviltà: vanno cacciate con ogni mezzo dal nostro suolo patrio."

#### Esempi per `negativity`:
- **low**:
  - "I dati sull'occupazione sono incoraggianti e la crescita economica dimostra la resilienza del nostro tessuto produttivo."
  - "La riforma della giustizia procede spedita, restituendo efficienza e tempi certi a cittadini e imprese."
  - "Il nostro posizionamento internazionale si rafforza grazie a nuove partnership strategiche sull'energia."
- **mid**:
  - "I sistemi sanitario e scolastico presentano problematiche infrastrutturali apparentemente insanabili."
  - "L'instabilità geopolitica globale proietta ombre preoccupanti sull'approvvigionamento delle nostre materie prime."
  - "La burocrazia soffocante e i tempi della giustizia civile continuano a frenare gli investimenti esteri nel Paese."
- **high**:
  - "Il paese è sull'orlo del baratro finanziario; ereditiamo un disastro sistemico che rischia di spazzare via i risparmi degli italiani."
  - "Siamo di fronte a un declino demografico e sociale irreversibile che sta portando la Nazione alla completa estinzione."
  - "Il collasso della sicurezza urbana ha trasformato le nostre metropoli in zone di guerra allo sbaraglio."

#### Esempi per `aggressiveness`:
- **low**:
  - "Accogliamo le osservazioni delle opposizioni nel merito, tuttavia il Governo continuerà con la linea dettata dagli elettori."
  - "Il confronto parlamentare è la sede naturale per affinare i testi di legge nell'interesse generale."
  - "Valuteremo con attenzione tutti gli emendamenti proposti dalle forze di minoranza durante il percorso in commissione."
- **mid**:
  - "Riconosco il diritto dell'opposizione di protestare, anche se la loro memoria storica appare alquanto corta e di comodo."
  - "Ci danno lezioni di rigore di bilancio gli stessi banchieri e tecnocrati che hanno affossato i conti pubblici nel decennio scorso."
  - "Spiace constatare come parte dei media preferisca la polemica strumentale all'analisi obiettiva dei fatti."
- **high**:
  - "Dall'opposizione arrivano solo menzogne sfrontate e sciacallaggio politico da parte di chi ha svenduto la nazione per anni."
  - "Siete dei traditori del popolo italiano, dei cospiratori servili che prendono ordini da potenze e burocrazie straniere."
  - "La vostra ipocrisia fa schifo: avete le mani sporche di sangue per le politiche criminali che avete approvato!"

#### Esempi per `target`:
- **none**:
  - "Oggi approviamo la riforma del codice della strada per ridurre gli incidenti e tutelare i giovani."
  - "Il piano di digitalizzazione della pubblica amministrazione consentirà di ridurre radicalmente i tempi di attesa."
  - "I finanziamenti per la prevenzione del dissesto idrogeologico copriranno tutte le regioni a rischio."
- **pol_adv**:
  - "La precedente maggioranza ha lasciato buchi di bilancio incalcolabili per fare propaganda elettorale."
  - "L'Unione Europea pretende di imporre direttive ideologiche che penalizzano le nostre filiere produttive nazionali."
  - "La magistratura politicamente orientata continua a travalicare i propri confini costituzionali per ostacolare l'esecutivo."
- **minor_etn**:
  - "È necessario bloccare le partenze e combattere le reti di clandestinità che destabilizzano la sicurezza nazionale."
  - "Certi flussi migratori incontrollati provenienti dall'Africa subsahariana importano modelli criminali inaccettabili."
  - "Non tollereremo zone di franchigia gestite da bande etniche straniere nelle periferie dei nostri capoluoghi."
- **minor_gnd**:
  - "Ci opporremo a chi vuole scardinare la famiglia naturale imponendo teorie ideologiche nelle scuole."
  - "La propaganda sull'identità di genere mira a cancellare la figura della madre e il ruolo biologico della donna."
  - "I diritti delle donne vengono calpestati dall'ossessione per il politicamente corretto e la fluidità di genere."
- **minor_rel**:
  - "Alcune comunità religiose pretendono di applicare le proprie leggi teocratiche sul nostro territorio nazionale."
  - "Il proliferare di centri di preghiera abusivi legati all'Islam radicale rappresenta una minaccia diretta ai nostri valori laici."
  - "Non faremo concessioni a chi usa il proprio culto per giustificare la sottomissione femminile e l'odio verso l'Occidente."

---
### FORMATO DI OUTPUT

Rispondi ESCLUSIVAMENTE con un oggetto JSON con queste quattro chiavi, senza testo
aggiuntivo, markdown o spiegazioni."""

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "hate_speech": {"type": "string", "enum": LEVEL_VALUES},
        "negativity": {"type": "string", "enum": LEVEL_VALUES},
        "aggressiveness": {"type": "string", "enum": LEVEL_VALUES},
        "target": {"type": "string", "enum": TARGET_VALUES},
    },
    "required": ["hate_speech", "negativity", "aggressiveness", "target"],
}


# --------------------------------------------------------------------------- #
# File discovery
# --------------------------------------------------------------------------- #
def find_speech_files() -> list:
    files = []
    for pol in POLITICIANS_TO_PROCESS:
        if pol == 'degasperi':
            pattern = os.path.join(PMS_DIR, pol, "degasperi_speeches.csv")
        else:
            pattern = os.path.join(PMS_DIR, pol, "csv_out", "*.csv")
        matches = sorted(glob.glob(pattern))
        if not matches:
            continue
        files.extend(matches)
    return files

# --------------------------------------------------------------------------- #
# Row helpers
# --------------------------------------------------------------------------- #

def needs_classification(row: dict) -> bool:
    """True if any of the 4 label columns is missing from the row, or empty/blank."""
    for col in LABEL_COLS:
        val = row.get(col)
        if val is None:
            return True
        if str(val).strip() == "":
            return True
    return False


def read_speech_csv(path: str):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames or [])
    return rows, fieldnames


def write_speech_csv(path: str, rows: list, fieldnames: list):
    # ensure the 4 label columns are present in the header (appended at the
    # end if they weren't already there), everything else preserved as-is
    out_fieldnames = list(fieldnames)
    for col in LABEL_COLS:
        if col not in out_fieldnames:
            out_fieldnames.append(col)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=out_fieldnames)
        writer.writeheader()
        writer.writerows(rows)


# --------------------------------------------------------------------------- #
# Classification
# --------------------------------------------------------------------------- #

def classify_one(text: str, path: str) -> dict | None:
    for attempt in range(1, RETRIES + 1):
        if VERBOSE:
            preview = text if PRINT_TEXT_CHARS is None else text[:PRINT_TEXT_CHARS]
            truncated_note = "" if PRINT_TEXT_CHARS is None or len(text) <= PRINT_TEXT_CHARS else " [truncated]"
            print(f"\n{'=' * 80}")
            print(f"[qwen2.5-7b] {path}  attempt {attempt}/{RETRIES}")
            print(f"--- PROMPT FED TO MODEL ({len(text)} chars{truncated_note}) ---")
            print(preview)

        try:
            resp = client.chat(
                model=OLLAMA_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                format=JSON_SCHEMA,
                options={"temperature": 0},
            )
            raw = resp["message"]["content"]

            if VERBOSE:
                print(f"--- RAW MODEL OUTPUT ---\n{raw}")

            parsed = json.loads(raw)
            # validate values are in-range; raises if not
            assert parsed["hate_speech"] in LEVEL_VALUES
            assert parsed["negativity"] in LEVEL_VALUES
            assert parsed["aggressiveness"] in LEVEL_VALUES
            assert parsed["target"] in TARGET_VALUES

            if VERBOSE:
                print(f"--- PARSED PREDICTION --- {parsed}")

            return parsed
        except Exception as e:
            print(f"  [qwen2.5-7b] {path} attempt {attempt}/{RETRIES} failed: {e}")
            time.sleep(RETRY_SLEEP_S)
    return None


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    files = find_speech_files()
    print(f"Found {len(files)} speech files under csv_out/ across {len(POLITICIANS_TO_PROCESS)} politicians.")

    n_skipped_already = 0
    n_skipped_no_text = 0
    n_classified = 0
    n_failed = 0
    n_files_saved = 0

    for path in files:
        try:
            rows, fieldnames = read_speech_csv(path)
        except Exception as e:
            print(f"  [SKIP] could not read {path}: {e}")
            continue

        if not rows:
            print(f"  [SKIP] {path} is empty.")
            continue

        file_changed = False

        for row in rows:
            if not needs_classification(row):
                n_skipped_already += 1
                continue

            text = (row.get("text") or "").strip()
            if not text:
                print(f"  [SKIP] {path}: empty '{"text"}' field, cannot classify.")
                n_skipped_no_text += 1
                continue

            result = classify_one(text, path)
            if result is None:
                print(f"  [FAIL] giving up on {path} after {RETRIES} retries.")
                n_failed += 1
                continue

            row.update(result)
            file_changed = True
            n_classified += 1

        if file_changed:
            write_speech_csv(path, rows, fieldnames)
            n_files_saved += 1
            print(f"  [SAVED] {path}")

    print("\n===== SUMMARY =====")
    print(f"Files scanned:           {len(files)}")
    print(f"Files rewritten:         {n_files_saved}")
    print(f"Speeches classified:     {n_classified}")
    print(f"Already labeled (skip):  {n_skipped_already}")
    print(f"Empty text (skip):       {n_skipped_no_text}")
    print(f"Failed after retries:    {n_failed}")


if __name__ == "__main__":
    main()

Found 1 speech files under csv_out/ across 1 politicians.

[qwen2.5-7b] /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/gentiloni/csv_out/702281_gentiloni_s2t.csv  attempt 1/3
--- PROMPT FED TO MODEL (3674 chars [truncated]) ---
La Commissione oggi ha dettato il quadro giuridico per l'introduzione dell'Eurodigitale. Il messaggio è semplice, non possiamo immaginare in un futuro digitale la moneta dominata soltanto da privati, da criptovalute e l'assenza di un ruolo per gli Stati, per la sovranità monetaria. E quindi anche in un mondo completamente diverso come il mondo digitale abbiamo bisogno di una moneta basata sulla so
--- RAW MODEL OUTPUT ---
{
  "hate_speech": "low",
  "negativity": "low",
  "aggressiveness": "low",
  "target": "none"
}
--- PARSED PREDICTION --- {'hate_speech': 'low', 'negativity': 'low', 'aggressiveness': 'low', 'target': 'none'}
  [SAVED] /home/mhetac/Documents/GitHub/crossdem/datasets/prime_ministers/gentiloni/csv_out/702281_gentiloni_s2t.csv

=

## Sentiment Aggregated L/R/C Between 1945-2025

## Sentiment 3 Axis

## Target Leaderboards 
Aggregated either by politician, political alignment or 5-year period